# P2_G0 계열별 기업유형·산업유형·자격증·초임급여·직무별 자격증 EDA

목표는 다섯 개 Excel을 각각 독립 DataFrame으로 먼저 확정한 뒤, 결합 가능한 분석 단위와 타겟 컬럼을 설계하는 것이다.

- 각 원본 파일별 대응 셀에서 `read_excel` → 행/컬럼 정리 → 숫자 변환 → 기술통계를 확인한다.
- 계열 기준 자료 4개는 `고등교육기관 x 계열` 단위로 통합 샘플을 만든다.
- `직무별 자격증 취득수`는 축이 `직무분류`라서 계열 자료와 직접 조인하지 않고, 자격증 시장의 직무별 집중도 보조 EDA로 둔다.
- 타겟 후보는 초임급여의 `평균소득만원`을 1순위, `중위소득만원`을 보조 타겟으로 설계한다.


In [1]:

from pathlib import Path
import os
import re
import warnings

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

MPLCONFIG_DIR = Path("/tmp/matplotlib-sbs-datascience")
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

PROJECT_DIR = Path("/home/sieg/projects-wsl/SBS_dataScience")
BASE_DIR = PROJECT_DIR / "workbook/p2/p2_2/final"
DATA_DIR = BASE_DIR / "data"
NOTEBOOK_PATH = BASE_DIR / "eda/P2_G0_계열별 기업, 산업, 자격증, 초임급여, 직무별.IPYNB"

FILES = {
    "company": DATA_DIR / "2024_계열별 기업유형별 취업현황.xlsx",
    "cert_major": DATA_DIR / "2024_계열별 자격증 취득현황.xlsx",
    "salary": DATA_DIR / "2024_계열별 초임급여 현황.xlsx",
    "industry": DATA_DIR / "2024_계열별 산업유형별 취업현황.xlsx",
    "cert_job": DATA_DIR / "2024_직무별 자격증 취득수.xlsx",
}

MAJORS = ["인문", "사회", "교육", "공학", "자연", "의약", "예체능"]
MAJOR_WITH_TOTAL = ["전체", *MAJORS]
MAIN_GROUPS = ["고등교육기관", "학부", "대학원"]
SALARY_GROUPS = ["고등교육기관", "학부", "대학원", "(석사)", "(박사)"]

KOREAN_FONT_FILES = [
    Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
    Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
    Path("/usr/share/fonts/opentype/noto/NotoSansCJKkr-Regular.otf"),
]
FONT_CANDIDATES = ["NanumGothic", "Noto Sans CJK KR", "Malgun Gothic", "AppleGothic", "DejaVu Sans"]


def configure_korean_font() -> str:
    for font_path in KOREAN_FONT_FILES:
        if font_path.exists():
            try:
                fm.fontManager.addfont(str(font_path))
            except RuntimeError:
                pass
    available = {font.name for font in fm.fontManager.ttflist}
    selected = next((font for font in FONT_CANDIDATES if font in available), "DejaVu Sans")
    mpl.rcParams["font.family"] = [selected]
    mpl.rcParams["font.sans-serif"] = [selected, *mpl.rcParams.get("font.sans-serif", [])]
    mpl.rcParams["axes.unicode_minus"] = False
    return selected


KOREAN_FONT = configure_korean_font()


def clean_column_name(value: object) -> str:
    text = str(value).replace("\ufeff", "").replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def clean_text(value: object) -> object:
    if pd.isna(value):
        return pd.NA
    text = str(value).replace("\ufeff", "").replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return pd.NA if text in {"", "nan", "None"} else text


def numericize(df: pd.DataFrame, text_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    text_cols = [col for col in text_cols if col in out.columns]
    for col in text_cols:
        out[col] = out[col].map(clean_text).astype("string")
    audit_rows = []
    for col in out.columns:
        if col in text_cols:
            continue
        before = out[col].notna()
        converted = pd.to_numeric(out[col], errors="coerce")
        audit_rows.append(
            {
                "column": col,
                "raw_non_null": int(before.sum()),
                "converted_non_null": int(converted.notna().sum()),
                "coerced_to_na": int((before & converted.isna()).sum()),
            }
        )
        out[col] = converted
    return out, pd.DataFrame(audit_rows)


def read_table(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="Sheet0", dtype=object, engine="openpyxl")
    df.columns = [clean_column_name(col) for col in df.columns]
    df = df.dropna(how="all").copy()
    return df


def source_overview(name: str, df: pd.DataFrame, key_cols: list[str], numeric_cols: list[str] | None = None) -> pd.DataFrame:
    numeric_cols = numeric_cols or df.select_dtypes(include="number").columns.tolist()
    duplicate_key_rows = int(df.duplicated(key_cols, keep=False).sum()) if all(col in df.columns for col in key_cols) else pd.NA
    return pd.DataFrame(
        [
            {"metric": "dataset", "value": name, "note": "DataFrame variable is fixed in this notebook"},
            {"metric": "rows", "value": len(df), "note": "filtered analysis rows only"},
            {"metric": "columns", "value": df.shape[1], "note": "after column-name cleanup"},
            {"metric": "numeric_columns", "value": len(numeric_cols), "note": "columns converted with pd.to_numeric"},
            {"metric": "duplicate_key_rows", "value": duplicate_key_rows, "note": "+".join(key_cols)},
        ]
    )


def display_basic_eda(title: str, df: pd.DataFrame, key_cols: list[str], conversion_audit: pd.DataFrame | None = None) -> None:
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    display(Markdown(f"## {title}"))
    display(source_overview(title, df, key_cols, numeric_cols))
    if conversion_audit is not None:
        problems = conversion_audit[conversion_audit["coerced_to_na"] > 0].copy()
        display(Markdown("### 숫자 변환 점검"))
        display(problems if len(problems) else pd.DataFrame([{"status": "숫자 변환 실패 없음"}]))
    display(Markdown("### DataFrame 샘플"))
    display(df.head(12))
    display(Markdown("### 수치형 기술통계"))
    display(df[numeric_cols].describe().T if numeric_cols else pd.DataFrame([{"status": "수치형 컬럼 없음"}]))
    display(Markdown("### 결측/고유값 프로파일"))
    profile = pd.DataFrame(
        [
            {
                "column": col,
                "dtype": str(df[col].dtype),
                "missing_n": int(df[col].isna().sum()),
                "missing_pct": float(df[col].isna().mean() * 100),
                "unique_n": int(df[col].nunique(dropna=True)),
            }
            for col in df.columns
        ]
    )
    display(profile)


def add_share_columns(df: pd.DataFrame, denominator: str, count_cols: list[str], suffix: str = "_비율") -> pd.DataFrame:
    out = df.copy()
    denom = pd.to_numeric(out[denominator], errors="coerce").replace(0, np.nan)
    for col in count_cols:
        out[f"{col}{suffix}"] = pd.to_numeric(out[col], errors="coerce") / denom * 100
    return out


display(Markdown(f"""
## 0. 공통 설정

- 데이터 폴더: `{DATA_DIR}`
- 대상 노트북: `{NOTEBOOK_PATH}`
- 계열 결합 후보: `{', '.join(MAJORS)}`
- Matplotlib 한글 폰트: `{KOREAN_FONT}`
"""))
for key, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(path)
    display(Markdown(f"- `{key}`: `{path.name}` ({path.stat().st_size:,} bytes)"))



## 0. 공통 설정

- 데이터 폴더: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/data`
- 대상 노트북: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0_계열별 기업, 산업, 자격증, 초임급여, 직무별.IPYNB`
- 계열 결합 후보: `인문, 사회, 교육, 공학, 자연, 의약, 예체능`
- Matplotlib 한글 폰트: `NanumGothic`


- `company`: `2024_계열별 기업유형별 취업현황.xlsx` (6,630 bytes)

- `cert_major`: `2024_계열별 자격증 취득현황.xlsx` (5,840 bytes)

- `salary`: `2024_계열별 초임급여 현황.xlsx` (7,429 bytes)

- `industry`: `2024_계열별 산업유형별 취업현황.xlsx` (6,013 bytes)

- `cert_job`: `2024_직무별 자격증 취득수.xlsx` (5,268 bytes)

In [2]:

raw_company = read_table(FILES["company"])
df_company = raw_company[
    raw_company["구분"].map(clean_text).isin(MAIN_GROUPS)
    & raw_company["계열"].map(clean_text).isin(MAJOR_WITH_TOTAL)
].copy()
df_company, company_conversion_audit = numericize(df_company, text_cols=["구분", "계열"])

company_count_cols = ["대기업", "중견기업", "중소기업", "국가및지방", "공공기관", "비영리", "기타"]
df_company = add_share_columns(df_company, "분석대상자", company_count_cols)
df_company["대중견기업비율"] = (df_company["대기업"] + df_company["중견기업"]) / df_company["분석대상자"].replace(0, np.nan) * 100
df_company["공공비영리비율"] = (df_company["국가및지방"] + df_company["공공기관"] + df_company["비영리"]) / df_company["분석대상자"].replace(0, np.nan) * 100

company_by_group = df_company.groupby("구분")["계열"].nunique(dropna=True).reset_index(name="계열수")
display_basic_eda("1. df_company: 계열별 기업유형별 취업현황", df_company, key_cols=["구분", "계열"], conversion_audit=company_conversion_audit)
display(Markdown("### 구분별 계열 수"))
display(company_by_group)
display(Markdown("### 고등교육기관 계열별 기업유형 비율"))
display(df_company[(df_company["구분"] == "고등교육기관") & (df_company["계열"].isin(MAJORS))][["계열", "대기업_비율", "중견기업_비율", "중소기업_비율", "대중견기업비율", "공공비영리비율"]])


## 1. df_company: 계열별 기업유형별 취업현황

,metric,value,note
0,dataset,1. df_company: 계열별 기업유형별 취업현황,DataFrame variable is fixed in this notebook
1,rows,24,filtered analysis rows only
2,columns,19,after column-name cleanup
3,numeric_columns,17,columns converted with pd.to_numeric
4,duplicate_key_rows,0,구분+계열


### 숫자 변환 점검

,status
0,숫자 변환 실패 없음


### DataFrame 샘플

...

,column,dtype,missing_n,missing_pct,unique_n
0,구분,string,0,0.000,3
1,계열,string,0,0.000,8
2,분석대상자,int64,0,0.000,24
3,대기업,int64,0,0.000,24
4,중견기업,int64,0,0.000,24
5,중소기업,int64,0,0.000,24
6,국가및지방,int64,0,0.000,24
7,공공기관,int64,0,0.000,24
8,비영리,int64,0,0.000,24
9,기타,int64,0,0.000,24


### 구분별 계열 수

,구분,계열수
0,고등교육기관,8
1,대학원,8
2,학부,8


### 고등교육기관 계열별 기업유형 비율

,계열,대기업_비율,중견기업_비율,중소기업_비율,대중견기업비율,공공비영리비율
1,인문,11.065,5.513,41.176,16.578,38.627
2,사회,12.164,6.290,40.393,18.453,35.060
3,교육,1.965,0.953,13.159,2.918,61.120
4,공학,23.100,10.385,49.729,33.485,16.060
5,자연,14.536,7.071,50.670,21.608,26.102
6,의약,1.677,1.064,46.397,2.742,50.343
7,예체능,7.959,4.423,69.328,12.382,17.327


In [3]:

raw_cert_major = read_table(FILES["cert_major"])
df_cert_major = raw_cert_major[
    raw_cert_major["구분"].map(clean_text).isin(MAIN_GROUPS)
    & raw_cert_major["계열"].map(clean_text).isin(MAJOR_WITH_TOTAL)
].copy()
df_cert_major, cert_major_conversion_audit = numericize(df_cert_major, text_cols=["구분", "계열"])

if "자격증 취득률" in df_cert_major.columns:
    df_cert_major["자격증취득률_계산값"] = df_cert_major["자격증취득자"] / df_cert_major["분석대상자"].replace(0, np.nan) * 100
    df_cert_major["자격증취득률_차이"] = df_cert_major["자격증 취득률"] - df_cert_major["자격증취득률_계산값"]

cert_major_by_group = df_cert_major.groupby("구분")["계열"].nunique(dropna=True).reset_index(name="계열수")
display_basic_eda("2. df_cert_major: 계열별 자격증 취득현황", df_cert_major, key_cols=["구분", "계열"], conversion_audit=cert_major_conversion_audit)
display(Markdown("### 구분별 계열 수"))
display(cert_major_by_group)
display(Markdown("### 고등교육기관 계열별 자격증 취득률/취득수"))
display(df_cert_major[(df_cert_major["구분"] == "고등교육기관") & (df_cert_major["계열"].isin(MAJORS))][["계열", "분석대상자", "자격증취득자", "자격증 취득률", "자격증취득수", "1인당 자격증 취득수"]])


## 2. df_cert_major: 계열별 자격증 취득현황

,metric,value,note
0,dataset,2. df_cert_major: 계열별 자격증 취득현황,DataFrame variable is fixed in this notebook
1,rows,24,filtered analysis rows only
2,columns,9,after column-name cleanup
3,numeric_columns,7,columns converted with pd.to_numeric
4,duplicate_key_rows,0,구분+계열


### 숫자 변환 점검

,status
0,숫자 변환 실패 없음


### DataFrame 샘플

...

,column,dtype,missing_n,missing_pct,unique_n
0,구분,string,0,0.000,3
1,계열,string,0,0.000,8
2,분석대상자,int64,0,0.000,24
3,자격증취득자,int64,0,0.000,24
4,자격증 취득률,float64,0,0.000,24
5,자격증취득수,int64,0,0.000,24
6,1인당 자격증 취득수,float64,0,0.000,10
7,자격증취득률_계산값,float64,0,0.000,24
8,자격증취득률_차이,float64,0,0.000,24


### 구분별 계열 수

,구분,계열수
0,고등교육기관,8
1,대학원,8
2,학부,8


### 고등교육기관 계열별 자격증 취득률/취득수

,계열,분석대상자,자격증취득자,자격증 취득률,자격증취득수,1인당 자격증 취득수
1,인문,23543,10101,42.900,17997,1.800
2,사회,82516,37707,45.700,70748,1.900
3,교육,25602,9683,37.800,18580,1.900
4,공학,86625,52824,61.000,124758,2.400
5,자연,28212,16456,58.300,35351,2.100
6,의약,47637,13201,27.700,22329,1.700
7,예체능,25117,8341,33.200,13684,1.600


In [4]:

raw_salary = read_table(FILES["salary"])
# 원본은 상단 2행 헤더 구조 때문에 pandas가 마지막 두 컬럼을 같은 이름으로 읽는다.
# 첫 데이터 행에도 헤더 보조행이 반복되므로, 그 행을 제거하고 평균/중위 컬럼명을 명확히 바꾼다.
df_salary = raw_salary.rename(columns={"월 평균 소득액": "평균소득만원", "월 평균 소득액.1": "중위소득만원"}).copy()
df_salary = df_salary[
    df_salary["구분"].map(clean_text).isin(SALARY_GROUPS)
    & df_salary["계열"].map(clean_text).isin(["전체", "소계", *MAJORS])
].copy()
df_salary, salary_conversion_audit = numericize(df_salary, text_cols=["구분", "계열"])

salary_bucket_cols = ["100만원미만", "100~200만원미만", "200~300만원미만", "300~400만원미만", "400만원이상"]
df_salary = add_share_columns(df_salary, "분석대상자", salary_bucket_cols)
df_salary["300만원이상_비율"] = (df_salary["300~400만원미만"] + df_salary["400만원이상"]) / df_salary["분석대상자"].replace(0, np.nan) * 100
df_salary["200만원미만_비율"] = (df_salary["100만원미만"] + df_salary["100~200만원미만"]) / df_salary["분석대상자"].replace(0, np.nan) * 100
df_salary["평균_중위_차이"] = df_salary["평균소득만원"] - df_salary["중위소득만원"]
df_salary["평균_중위_비율"] = df_salary["평균소득만원"] / df_salary["중위소득만원"].replace(0, np.nan)

salary_by_group = df_salary.groupby("구분")["계열"].nunique(dropna=True).reset_index(name="계열수")
display_basic_eda("3. df_salary: 계열별 초임급여 현황", df_salary, key_cols=["구분", "계열"], conversion_audit=salary_conversion_audit)
display(Markdown("### 구분별 계열 수"))
display(salary_by_group)
display(Markdown("### 고등교육기관 계열별 초임급여 타겟 후보"))
display(df_salary[(df_salary["구분"] == "고등교육기관") & (df_salary["계열"].isin(MAJORS))][["계열", "평균소득만원", "중위소득만원", "평균_중위_차이", "300만원이상_비율", "400만원이상_비율", "200만원미만_비율"]])


## 3. df_salary: 계열별 초임급여 현황

,metric,value,note
0,dataset,3. df_salary: 계열별 초임급여 현황,DataFrame variable is fixed in this notebook
1,rows,40,filtered analysis rows only
2,columns,19,after column-name cleanup
3,numeric_columns,17,columns converted with pd.to_numeric
4,duplicate_key_rows,0,구분+계열


### 숫자 변환 점검

,status
0,숫자 변환 실패 없음


### DataFrame 샘플

...

,column,dtype,missing_n,missing_pct,unique_n
0,구분,string,0,0.000,5
1,계열,string,0,0.000,9
2,분석대상자,int64,0,0.000,40
3,100만원미만,int64,0,0.000,38
4,100~200만원미만,int64,0,0.000,39
5,200~300만원미만,int64,0,0.000,39
6,300~400만원미만,int64,0,0.000,40
7,400만원이상,int64,0,0.000,40
8,평균소득만원,float64,0,0.000,40
9,중위소득만원,float64,0,0.000,37


### 구분별 계열 수

,구분,계열수
0,(박사),8
1,(석사),8
2,고등교육기관,8
3,대학원,8
4,학부,8


### 고등교육기관 계열별 초임급여 타겟 후보

,계열,평균소득만원,중위소득만원,평균_중위_차이,300만원이상_비율,400만원이상_비율,200만원미만_비율
2,인문,316.300,255.900,60.400,36.079,20.074,15.440
3,사회,364.300,270.700,93.600,41.954,25.910,12.155
4,교육,329.700,290.600,39.100,46.699,25.205,10.804
5,공학,361.700,308.300,53.400,52.930,30.570,7.086
6,자연,311.000,260.000,51.000,35.403,17.528,12.924
7,의약,360.200,294.400,65.800,48.359,19.434,6.335
8,예체능,245.100,220.000,25.100,16.415,7.087,26.679


In [5]:

raw_industry = read_table(FILES["industry"])
df_industry, industry_conversion_audit = numericize(raw_industry, text_cols=["산업분류"])
# 주석 행은 분석대상자/계열별 수치가 숫자로 변환되지 않는다. 숫자 행만 남기면 전체/산업/기타/결측값 행이 보존된다.
df_industry = df_industry[df_industry["분석대상자"].notna()].copy()

industry_value_cols = [col for col in MAJORS if col in df_industry.columns]
df_industry_long = df_industry.melt(
    id_vars=["산업분류", "분석대상자"],
    value_vars=industry_value_cols,
    var_name="계열",
    value_name="산업취업자수",
)
industry_totals = df_industry[df_industry["산업분류"] == "전체"].set_index("산업분류")[industry_value_cols].iloc[0]
df_industry_long["계열전체취업자수"] = df_industry_long["계열"].map(industry_totals.to_dict())
df_industry_long["산업비율"] = df_industry_long["산업취업자수"] / df_industry_long["계열전체취업자수"].replace(0, np.nan) * 100

industry_no_total = df_industry_long[~df_industry_long["산업분류"].isin(["전체", "결측값"])].copy()
industry_top_by_major = (
    industry_no_total.sort_values(["계열", "산업취업자수"], ascending=[True, False])
    .groupby("계열")
    .head(3)
    .loc[:, ["계열", "산업분류", "산업취업자수", "산업비율"]]
    .reset_index(drop=True)
)

display_basic_eda("4. df_industry: 계열별 산업유형별 취업현황", df_industry, key_cols=["산업분류"], conversion_audit=industry_conversion_audit)
display(Markdown("### df_industry_long 샘플"))
display(df_industry_long.head(20))
display(Markdown("### 계열별 상위 산업 Top 3"))
display(industry_top_by_major)
display(Markdown("### 산업비율 기술통계"))
display(df_industry_long[df_industry_long["산업분류"] != "전체"].groupby("계열")["산업비율"].describe())


## 4. df_industry: 계열별 산업유형별 취업현황

,metric,value,note
0,dataset,4. df_industry: 계열별 산업유형별 취업현황,DataFrame variable is fixed in this notebook
1,rows,22,filtered analysis rows only
2,columns,9,after column-name cleanup
3,numeric_columns,8,columns converted with pd.to_numeric
4,duplicate_key_rows,0,산업분류


### 숫자 변환 점검

,status
0,숫자 변환 실패 없음


### DataFrame 샘플

...

,산업분류,분석대상자,계열,산업취업자수,계열전체취업자수,산업비율
0,전체,"319,252.000",인문,"23,543.000","23,543.000",100.000
1,A.농림어업,497.000,인문,20.000,"23,543.000",0.085
2,B.광업,47.000,인문,2.000,"23,543.000",0.008
3,C.제조업,"58,482.000",인문,"2,510.000","23,543.000",10.661
4,D.전기·가스·증기업,"1,476.000",인문,59.000,"23,543.000",0.251
5,E.수도·하수·폐기물업,744.000,인문,39.000,"23,543.000",0.166
6,F.건설업,"10,922.000",인문,443.000,"23,543.000",1.882
7,G.도·소매업,"21,716.000",인문,"2,255.000","23,543.000",9.578
8,H.운수업,"7,937.000",인문,"1,010.000","23,543.000",4.290
9,I.숙박·음식점업,"11,003.000",인문,864.000,"23,543.000",3.670


### 계열별 상위 산업 Top 3

,계열,산업분류,산업취업자수,산업비율
0,공학,C.제조업,"33,024.000",38.123
1,공학,M.전문·과학·기술업,"11,516.000",13.294
2,공학,J.정보통신업,"10,670.000",12.317
3,교육,P.교육서비스업,"11,144.000",43.528
4,교육,Q.보건·사회복지업,"5,307.000",20.729
5,교육,O.공공행정,"2,674.000",10.444
6,사회,Q.보건·사회복지업,"10,874.000",13.178
7,사회,O.공공행정,"10,762.000",13.042
8,사회,C.제조업,"10,120.000",12.264
9,예체능,G.도·소매업,"3,431.000",13.660


### 산업비율 기술통계

,count,mean,std,min,25%,50%,75%,max
계열,,,,,,,,
공학,21.000,4.762,8.555,0.023,0.776,1.463,5.473,38.123
교육,21.000,4.762,10.210,0.012,0.418,1.066,2.254,43.528
사회,21.000,4.762,4.400,0.018,1.138,4.156,7.077,13.178
예체능,21.000,4.762,4.368,0.008,0.864,4.770,7.907,13.660
의약,21.000,4.762,16.667,0.004,0.248,0.726,2.095,77.280
인문,21.000,4.762,4.309,0.008,1.083,3.670,7.947,13.350
자연,21.000,4.762,6.331,0.011,0.932,2.212,4.927,24.965


In [6]:

raw_cert_job = read_table(FILES["cert_job"])
df_cert_job, cert_job_conversion_audit = numericize(raw_cert_job, text_cols=["직무분류"])
df_cert_job = df_cert_job[df_cert_job["고등교육기관"].notna()].copy()

cert_job_value_cols = ["고등교육기관", "학부", "대학원"]
df_cert_job_long = df_cert_job.melt(
    id_vars=["직무분류"],
    value_vars=cert_job_value_cols,
    var_name="구분",
    value_name="자격증취득수",
)
cert_job_total = float(df_cert_job.loc[df_cert_job["직무분류"] == "전체", "고등교육기관"].iloc[0])
df_cert_job["고등교육기관_직무비율"] = df_cert_job["고등교육기관"] / cert_job_total * 100
cert_job_top = df_cert_job[df_cert_job["직무분류"] != "전체"].sort_values("고등교육기관", ascending=False).head(12)

display_basic_eda("5. df_cert_job: 직무별 자격증 취득수", df_cert_job, key_cols=["직무분류"], conversion_audit=cert_job_conversion_audit)
display(Markdown("### df_cert_job_long 샘플"))
display(df_cert_job_long.head(15))
display(Markdown("### 고등교육기관 기준 직무별 자격증 취득수 Top 12"))
display(cert_job_top[["직무분류", "고등교육기관", "학부", "대학원", "고등교육기관_직무비율"]])


## 5. df_cert_job: 직무별 자격증 취득수

,metric,value,note
0,dataset,5. df_cert_job: 직무별 자격증 취득수,DataFrame variable is fixed in this notebook
1,rows,24,filtered analysis rows only
2,columns,5,after column-name cleanup
3,numeric_columns,4,columns converted with pd.to_numeric
4,duplicate_key_rows,0,직무분류


### 숫자 변환 점검

,status
0,숫자 변환 실패 없음


### DataFrame 샘플

...

,column,dtype,missing_n,missing_pct,unique_n
0,직무분류,string,0,0.000,24
1,고등교육기관,float64,0,0.000,24
2,학부,float64,0,0.000,24
3,대학원,float64,0,0.000,24
4,고등교육기관_직무비율,float64,0,0.000,24


### df_cert_job_long 샘플

,직무분류,구분,자격증취득수
0,전체,고등교육기관,"303,447.000"
1,경영·회계·사무,고등교육기관,"127,952.000"
2,교육·자연과학·사회과학,고등교육기관,11.000
3,보건·의료,고등교육기관,938.000
4,사회복지·종교,고등교육기관,"1,247.000"
5,문화·예술·디자인·방송,고등교육기관,"3,931.000"
6,운전·운송,고등교육기관,238.000
7,영업·판매,고등교육기관,646.000
8,이용·숙박·여행·오락·스포츠,고등교육기관,"6,234.000"
9,음식서비스,고등교육기관,"13,790.000"


### 고등교육기관 기준 직무별 자격증 취득수 Top 12

,직무분류,고등교육기관,학부,대학원,고등교육기관_직무비율
1,경영·회계·사무,"127,952.000","92,349.000","35,603.000",42.166
10,건설,"30,144.000","24,665.000","5,479.000",9.934
17,정보통신,"27,102.000","17,560.000","9,542.000",8.931
12,기계,"24,490.000","21,767.000","2,723.000",8.071
16,전기·전자,"17,238.000","15,017.000","2,221.000",5.681
21,안전관리,"16,050.000","13,126.000","2,924.000",5.289
9,음식서비스,"13,790.000","11,524.000","2,266.000",4.544
14,화학,"7,837.000","6,512.000","1,325.000",2.583
18,식품가공,"7,228.000","6,133.000","1,095.000",2.382
8,이용·숙박·여행·오락·스포츠,"6,234.000","5,663.000",571.000,2.054


In [7]:

contract_rows = [
    {
        "df_name": "df_company",
        "source": FILES["company"].name,
        "analysis_unit": "구분 x 계열",
        "join_key": "구분, 계열",
        "rows": len(df_company),
        "target_candidate": "아니오",
        "main_feature_candidates": "대기업_비율, 대중견기업비율, 중소기업_비율, 공공비영리비율",
        "join_note": "기업유형 비율 피처로 사용",
    },
    {
        "df_name": "df_cert_major",
        "source": FILES["cert_major"].name,
        "analysis_unit": "구분 x 계열",
        "join_key": "구분, 계열",
        "rows": len(df_cert_major),
        "target_candidate": "보조 가능",
        "main_feature_candidates": "자격증 취득률, 1인당 자격증 취득수",
        "join_note": "계열별 역량/자격 신호 피처",
    },
    {
        "df_name": "df_salary",
        "source": FILES["salary"].name,
        "analysis_unit": "구분 x 계열",
        "join_key": "구분, 계열",
        "rows": len(df_salary),
        "target_candidate": "예",
        "main_feature_candidates": "평균소득만원, 중위소득만원, 300만원이상_비율, 200만원미만_비율",
        "join_note": "초임급여 타겟과 타겟 진단 컬럼",
    },
    {
        "df_name": "df_industry / df_industry_long",
        "source": FILES["industry"].name,
        "analysis_unit": "산업분류 x 계열; 구분 없음",
        "join_key": "계열",
        "rows": len(df_industry),
        "target_candidate": "아니오",
        "main_feature_candidates": "제조업비율, 정보통신업비율, 교육서비스비율, 보건사회복지비율, 상위산업비율",
        "join_note": "구분이 없으므로 고등교육기관 x 계열 통합 샘플에만 안전하게 결합",
    },
    {
        "df_name": "df_cert_job / df_cert_job_long",
        "source": FILES["cert_job"].name,
        "analysis_unit": "직무분류 x 구분",
        "join_key": "직무분류; 계열 없음",
        "rows": len(df_cert_job),
        "target_candidate": "별도 타겟 가능",
        "main_feature_candidates": "직무별 자격증취득수, 직무비율",
        "join_note": "계열과 직접 조인 불가. 직무-계열 매핑표가 있으면 확장 가능",
    },
]

df_data_contract = pd.DataFrame(contract_rows)

key_coverage = pd.DataFrame(
    [
        {"sample_candidate": "A_full_major", "unit": "고등교육기관 x 계열", "rows_expected": len(MAJORS), "included_sources": "company, cert_major, salary, industry", "excluded_or_context": "cert_job은 직무축이라 직접 조인 제외"},
        {"sample_candidate": "B_degree_major", "unit": "구분(고등교육기관/학부/대학원) x 계열", "rows_expected": len(MAIN_GROUPS) * len(MAJORS), "included_sources": "company, cert_major, salary", "excluded_or_context": "industry는 구분이 없어 제외, cert_job은 직무축"},
        {"sample_candidate": "C_job_cert", "unit": "직무분류 x 구분", "rows_expected": int(len(df_cert_job_long[df_cert_job_long['직무분류'] != '전체'])), "included_sources": "cert_job", "excluded_or_context": "계열 통합 샘플의 보조 설명 또는 후속 매핑 대상"},
    ]
)

display(Markdown("## 6. 통합 전 데이터 계약과 결합 가능성"))
display(df_data_contract)
display(Markdown("### 통합 샘플 후보"))
display(key_coverage)


## 6. 통합 전 데이터 계약과 결합 가능성

,df_name,source,analysis_unit,join_key,rows,target_candidate,main_feature_candidates,join_note
0,df_company,2024_계열별 기업유형별 취업현황.xlsx,구분 x 계열,"구분, 계열",24,아니오,"대기업_비율, 대중견기업비율, 중소기업_비율, 공공비영리비율",기업유형 비율 피처로 사용
1,df_cert_major,2024_계열별 자격증 취득현황.xlsx,구분 x 계열,"구분, 계열",24,보조 가능,"자격증 취득률, 1인당 자격증 취득수",계열별 역량/자격 신호 피처
2,df_salary,2024_계열별 초임급여 현황.xlsx,구분 x 계열,"구분, 계열",40,예,"평균소득만원, 중위소득만원, 300만원이상_비율, 200만원미만_비율",초임급여 타겟과 타겟 진단 컬럼
3,df_industry / df_industry_long,2024_계열별 산업유형별 취업현황.xlsx,산업분류 x 계열; 구분 없음,계열,22,아니오,"제조업비율, 정보통신업비율, 교육서비스비율, 보건사회복지비율, 상위산업비율",구분이 없으므로 고등교육기관 x 계열 통합 샘플에만 안전하게 결합
4,df_cert_job / df_cert_job_long,2024_직무별 자격증 취득수.xlsx,직무분류 x 구분,직무분류; 계열 없음,24,별도 타겟 가능,"직무별 자격증취득수, 직무비율",계열과 직접 조인 불가. 직무-계열 매핑표가 있으면 확장 가능


### 통합 샘플 후보

,sample_candidate,unit,rows_expected,included_sources,excluded_or_context
0,A_full_major,고등교육기관 x 계열,7,"company, cert_major, salary, industry",cert_job은 직무축이라 직접 조인 제외
1,B_degree_major,구분(고등교육기관/학부/대학원) x 계열,21,"company, cert_major, salary","industry는 구분이 없어 제외, cert_job은 직무축"
2,C_job_cert,직무분류 x 구분,69,cert_job,계열 통합 샘플의 보조 설명 또는 후속 매핑 대상


In [8]:

# A안: 네 개 계열축 파일을 고등교육기관 x 계열로 통합한다. 산업 파일은 구분이 없으므로 이 단위가 가장 안전하다.
company_major = df_company[(df_company["구분"] == "고등교육기관") & (df_company["계열"].isin(MAJORS))].copy()
company_major_features = company_major[
    [
        "계열",
        "대기업_비율",
        "중견기업_비율",
        "중소기업_비율",
        "국가및지방_비율",
        "공공기관_비율",
        "비영리_비율",
        "기타_비율",
        "대중견기업비율",
        "공공비영리비율",
    ]
].copy()

cert_major_high = df_cert_major[(df_cert_major["구분"] == "고등교육기관") & (df_cert_major["계열"].isin(MAJORS))].copy()
cert_major_features = cert_major_high[["계열", "자격증 취득률", "자격증취득자", "자격증취득수", "1인당 자격증 취득수"]].copy()

salary_major = df_salary[(df_salary["구분"] == "고등교육기관") & (df_salary["계열"].isin(MAJORS))].copy()
salary_major_features = salary_major[
    [
        "계열",
        "분석대상자",
        "평균소득만원",
        "중위소득만원",
        "평균_중위_차이",
        "평균_중위_비율",
        "100만원미만_비율",
        "100~200만원미만_비율",
        "200~300만원미만_비율",
        "300~400만원미만_비율",
        "400만원이상_비율",
        "300만원이상_비율",
        "200만원미만_비율",
    ]
].copy()

industry_feature_rows = []
industry_share_names = {
    "C.제조업": "제조업비율",
    "J.정보통신업": "정보통신업비율",
    "M.전문·과학·기술업": "전문과학기술비율",
    "O.공공행정": "공공행정비율",
    "P.교육서비스업": "교육서비스비율",
    "Q.보건·사회복지업": "보건사회복지비율",
    "R.예술·스포츠·여가업": "예술스포츠여가비율",
}
for major in MAJORS:
    g = industry_no_total[industry_no_total["계열"] == major].copy()
    g = g.sort_values("산업취업자수", ascending=False)
    top = g.iloc[0]
    row = {
        "계열": major,
        "상위산업": top["산업분류"],
        "상위산업비율": float(top["산업비율"]),
        "상위3산업비율": float(g.head(3)["산업비율"].sum()),
        "산업허핀달지수": float(((g["산업비율"] / 100) ** 2).sum()),
    }
    for raw_name, feature_name in industry_share_names.items():
        value = g.loc[g["산업분류"] == raw_name, "산업비율"]
        row[feature_name] = float(value.iloc[0]) if len(value) else 0.0
    industry_feature_rows.append(row)
industry_major_features = pd.DataFrame(industry_feature_rows)

df_integrated_major = (
    salary_major_features
    .merge(company_major_features, on="계열", how="left", validate="one_to_one")
    .merge(cert_major_features, on="계열", how="left", validate="one_to_one")
    .merge(industry_major_features, on="계열", how="left", validate="one_to_one")
)
df_integrated_major["log10_평균소득만원"] = np.log10(df_integrated_major["평균소득만원"])
df_integrated_major["log10_중위소득만원"] = np.log10(df_integrated_major["중위소득만원"])

# B안: 산업 없이 구분 x 계열까지 확장한 21행 샘플. 샘플 수는 늘지만 산업구조 설명력은 빠진다.
degree_salary = df_salary[(df_salary["구분"].isin(MAIN_GROUPS)) & (df_salary["계열"].isin(MAJORS))].copy()
degree_company = df_company[(df_company["구분"].isin(MAIN_GROUPS)) & (df_company["계열"].isin(MAJORS))].copy()
degree_cert = df_cert_major[(df_cert_major["구분"].isin(MAIN_GROUPS)) & (df_cert_major["계열"].isin(MAJORS))].copy()

df_integrated_degree_major = (
    degree_salary[["구분", "계열", "분석대상자", "평균소득만원", "중위소득만원", "300만원이상_비율", "200만원미만_비율", "평균_중위_차이"]]
    .merge(degree_company[["구분", "계열", "대기업_비율", "중소기업_비율", "대중견기업비율", "공공비영리비율"]], on=["구분", "계열"], how="left", validate="one_to_one")
    .merge(degree_cert[["구분", "계열", "자격증 취득률", "자격증취득수", "1인당 자격증 취득수"]], on=["구분", "계열"], how="left", validate="one_to_one")
)
df_integrated_degree_major["log10_평균소득만원"] = np.log10(df_integrated_degree_major["평균소득만원"])

display(Markdown("## 7. 통합 데이터 샘플 생성"))
display(Markdown("### A안: df_integrated_major = 고등교육기관 x 계열, 4개 계열축 자료 결합"))
display(df_integrated_major)
display(Markdown("### A안 기술통계"))
display(df_integrated_major.select_dtypes(include="number").describe().T)
display(Markdown("### B안: df_integrated_degree_major = 구분 x 계열, 산업 제외 21행 결합"))
display(df_integrated_degree_major.head(30))
display(Markdown("### B안 기술통계"))
display(df_integrated_degree_major.select_dtypes(include="number").describe().T)


## 7. 통합 데이터 샘플 생성

### A안: df_integrated_major = 고등교육기관 x 계열, 4개 계열축 자료 결합

,계열,분석대상자,평균소득만원,중위소득만원,평균_중위_차이,평균_중위_비율,100만원미만_비율,100~200만원미만_비율,200~300만원미만_비율,300~400만원미만_비율,400만원이상_비율,300만원이상_비율,200만원미만_비율,대기업_비율,중견기업_비율,중소기업_비율,국가및지방_비율,공공기관_비율,비영리_비율,기타_비율,대중견기업비율,공공비영리비율,자격증 취득률,자격증취득자,자격증취득수,1인당 자격증 취득수,상위산업,상위산업비율,상위3산업비율,산업허핀달지수,제조업비율,정보통신업비율,전문과학기술비율,공공행정비율,교육서비스비율,보건사회복지비율,예술스포츠여가비율,log10_평균소득만원,log10_중위소득만원
0,인문,23543,316.300,255.900,60.400,1.236,2.880,12.560,48.482,16.005,20.074,36.079,15.440,11.065,5.513,41.176,18.511,2.532,17.585,3.619,16.578,38.627,42.900,10101,17997,1.800,P.교육서비스업,13.350,36.669,0.084,10.661,8.317,7.947,12.658,13.350,7.595,2.094,2.500,2.408
1,사회,82516,364.300,270.700,93.600,1.346,2.197,9.958,45.890,16.044,25.910,41.954,12.155,12.164,6.290,40.393,14.910,3.839,16.311,6.093,18.453,35.060,45.700,37707,70748,1.900,Q.보건·사회복지업,13.178,38.485,0.086,12.264,5.895,9.950,13.042,5.346,13.178,1.138,2.561,2.432
2,교육,25602,329.700,290.600,39.100,1.135,1.578,9.226,42.497,21.494,25.205,46.699,10.804,1.965,0.953,13.159,41.618,0.980,18.522,22.803,2.918,61.120,37.800,9683,18580,1.900,P.교육서비스업,43.528,74.701,0.246,2.273,1.668,2.062,10.444,43.528,20.729,0.777,2.518,2.463
3,공학,86625,361.700,308.300,53.400,1.173,1.067,6.019,39.984,22.361,30.570,52.930,7.086,23.100,10.385,49.729,7.237,3.508,5.315,0.726,33.485,16.060,61.000,52824,124758,2.400,C.제조업,38.123,63.734,0.194,38.123,12.317,13.294,6.185,2.192,0.831,0.428,2.558,2.489
4,자연,28212,311.000,260.000,51.000,1.196,2.166,10.758,51.673,17.875,17.528,35.403,12.924,14.536,7.071,50.670,9.819,3.506,12.778,1.620,21.608,26.102,58.300,16456,35351,2.100,C.제조업,24.965,53.339,0.128,24.965,4.062,17.950,7.231,4.927,4.597,1.439,2.493,2.415
5,의약,47637,360.200,294.400,65.800,1.224,0.940,5.395,45.305,28.925,19.434,48.359,6.335,1.677,1.064,46.397,4.121,4.263,41.959,0.519,2.742,50.343,27.700,13201,22329,1.700,Q.보건·사회복지업,77.280,85.209,0.603,4.096,0.739,2.966,3.205,2.095,77.280,0.447,2.557,2.469
6,예체능,25117,245.100,220.000,25.100,1.114,4.531,22.148,56.906,9.328,7.087,16.415,26.679,7.959,4.423,69.328,6.517,1.063,9.746,0.963,12.382,17.327,33.200,8341,13684,1.600,G.도·소매업,13.660,36.461,0.085,12.947,9.854,8.576,4.790,9.424,4.770,4.993,2.389,2.342


### A안 기술통계

,count,mean,std,min,25%,50%,75%,max
분석대상자,7.000,"45,607.429","27,863.709","23,543.000","25,359.500","28,212.000","65,076.500","86,625.000"
평균소득만원,7.000,326.900,42.393,245.100,313.650,329.700,360.950,364.300
중위소득만원,7.000,271.414,29.642,220.000,257.950,270.700,292.500,308.300
평균_중위_차이,7.000,55.486,21.600,25.100,45.050,53.400,63.100,93.600
평균_중위_비율,7.000,1.203,0.077,1.114,1.154,1.196,1.230,1.346
100만원미만_비율,7.000,2.194,1.235,0.940,1.322,2.166,2.538,4.531
100~200만원미만_비율,7.000,10.866,5.584,5.395,7.622,9.958,11.659,22.148
200~300만원미만_비율,7.000,47.248,5.706,39.984,43.901,45.890,50.077,56.906
300~400만원미만_비율,7.000,18.862,6.170,9.328,16.024,17.875,21.928,28.925
400만원이상_비율,7.000,20.830,7.551,7.087,18.481,20.074,25.558,30.570


### B안: df_integrated_degree_major = 구분 x 계열, 산업 제외 21행 결합

,구분,계열,분석대상자,평균소득만원,중위소득만원,300만원이상_비율,200만원미만_비율,평균_중위_차이,대기업_비율,중소기업_비율,대중견기업비율,공공비영리비율,자격증 취득률,자격증취득수,1인당 자격증 취득수,log10_평균소득만원
0,고등교육기관,인문,23543,316.300,255.900,36.079,15.440,60.400,11.065,41.176,16.578,38.627,42.900,17997,1.800,2.500
1,고등교육기관,사회,82516,364.300,270.700,41.954,12.155,93.600,12.164,40.393,18.453,35.060,45.700,70748,1.900,2.561
2,고등교육기관,교육,25602,329.700,290.600,46.699,10.804,39.100,1.965,13.159,2.918,61.120,37.800,18580,1.900,2.518
3,고등교육기관,공학,86625,361.700,308.300,52.930,7.086,53.400,23.100,49.729,33.485,16.060,61.000,124758,2.400,2.558
4,고등교육기관,자연,28212,311.000,260.000,35.403,12.924,51.000,14.536,50.670,21.608,26.102,58.300,35351,2.100,2.493
5,고등교육기관,의약,47637,360.200,294.400,48.359,6.335,65.800,1.677,46.397,2.742,50.343,27.700,22329,1.700,2.557
6,고등교육기관,예체능,25117,245.100,220.000,16.415,26.679,25.100,7.959,69.328,12.382,17.327,33.200,13684,1.600,2.389
7,학부,인문,18844,303.900,250.000,32.482,15.819,53.900,12.816,45.553,19.290,31.846,41.600,13438,1.700,2.483
8,학부,사회,65600,300.300,250.500,32.473,13.832,49.800,11.988,42.675,18.636,32.723,44.500,52218,1.800,2.478
9,학부,교육,15498,260.200,235.300,27.184,14.692,24.900,2.407,15.931,3.549,48.722,25.100,7031,1.800,2.415


### B안 기술통계

,count,mean,std,min,25%,50%,75%,max
분석대상자,21.000,"30,404.952","25,879.540","2,304.000","13,849.000","22,813.000","41,094.000","86,625.000"
평균소득만원,21.000,365.929,111.399,234.700,303.900,329.700,366.000,648.500
중위소득만원,21.000,307.467,83.467,217.500,250.500,282.400,308.300,500.000
300만원이상_비율,21.000,46.896,20.561,13.427,32.482,46.007,52.930,85.696
200만원미만_비율,21.000,11.819,6.899,2.412,6.335,12.155,15.072,27.427
평균_중위_차이,21.000,58.462,40.262,13.100,32.900,53.400,66.000,198.500
대기업_비율,21.000,10.435,8.341,1.287,2.407,11.065,14.425,33.353
중소기업_비율,21.000,41.074,16.544,8.907,31.547,42.675,50.670,70.789
대중견기업비율,21.000,15.071,11.130,1.950,3.653,16.578,20.169,40.241
공공비영리비율,21.000,38.830,18.296,13.895,26.102,36.068,48.722,80.137


In [9]:

feature_cols_primary = [
    "대기업_비율",
    "중견기업_비율",
    "중소기업_비율",
    "국가및지방_비율",
    "공공기관_비율",
    "비영리_비율",
    "대중견기업비율",
    "공공비영리비율",
    "자격증 취득률",
    "1인당 자격증 취득수",
    "상위산업비율",
    "상위3산업비율",
    "산업허핀달지수",
    "제조업비율",
    "정보통신업비율",
    "전문과학기술비율",
    "공공행정비율",
    "교육서비스비율",
    "보건사회복지비율",
    "예술스포츠여가비율",
]
feature_cols_primary = [col for col in feature_cols_primary if col in df_integrated_major.columns]

target_candidates = pd.DataFrame(
    [
        {
            "target_column": "평균소득만원",
            "role": "primary_y",
            "why": "연속형이고 계열별 초임급여 수준 차이를 직접 보여준다.",
            "caveat": "n=7이면 모델링보다 가설 탐색용이다. 표본 확장은 학과/학교 단위 원자료가 필요하다.",
        },
        {
            "target_column": "log10_평균소득만원",
            "role": "primary_y_transformed_when_modeling",
            "why": "소득 타겟은 양수이고 상위 계열 영향이 커질 수 있어 곱셈적 차이를 완화한다.",
            "caveat": "현재 n=7에서는 변환 전후 순위가 거의 같으므로 설명용 보조 컬럼이다.",
        },
        {
            "target_column": "중위소득만원",
            "role": "secondary_y",
            "why": "평균이 일부 고소득자에 끌릴 때 전형적 졸업자 소득을 보완한다.",
            "caveat": "평균소득과 함께 보아야 하며 단독 타겟으로 쓰면 고소득 꼬리 정보를 잃는다.",
        },
        {
            "target_column": "300만원이상_비율 / 400만원이상_비율",
            "role": "distribution_target_or_segment_y",
            "why": "초임급여의 상위 구간 진입률을 설명할 때 기사형 타겟으로 해석하기 쉽다.",
            "caveat": "평균소득의 설명변수로 동시에 넣으면 타겟 누수에 가깝다. 별도 타겟으로만 사용한다.",
        },
        {
            "target_column": "자격증 취득률",
            "role": "alternate_y",
            "why": "취업성과가 아니라 계열별 자격 취득 행동 자체를 설명하려는 경우 타겟이 될 수 있다.",
            "caveat": "초임급여 EDA의 설명변수로 쓸 때와 역할을 분리한다.",
        },
    ]
)

candidate_columns = []
for col in ["평균소득만원", "중위소득만원", "log10_평균소득만원", "300만원이상_비율", "400만원이상_비율"]:
    candidate_columns.append({"column": col, "role": "target_or_target_diagnostic", "source": "df_salary", "use": "타겟 설계/급여 분포 진단"})
for col in ["대기업_비율", "중견기업_비율", "중소기업_비율", "대중견기업비율", "공공비영리비율"]:
    candidate_columns.append({"column": col, "role": "feature", "source": "df_company", "use": "계열별 취업처 규모/공공성 구조"})
for col in ["자격증 취득률", "1인당 자격증 취득수", "자격증취득수"]:
    candidate_columns.append({"column": col, "role": "feature_or_alternate_target", "source": "df_cert_major", "use": "자격 취득 강도와 초임급여 관계 후보"})
for col in ["상위산업", "상위산업비율", "상위3산업비율", "산업허핀달지수", "제조업비율", "정보통신업비율", "전문과학기술비율", "교육서비스비율", "보건사회복지비율"]:
    candidate_columns.append({"column": col, "role": "feature", "source": "df_industry_long", "use": "계열별 산업 집중도/산업 믹스"})
for col in ["직무분류", "고등교육기관", "고등교육기관_직무비율"]:
    candidate_columns.append({"column": col, "role": "context_or_future_feature", "source": "df_cert_job", "use": "직무-계열 매핑이 있을 때만 통합 피처화"})
candidate_columns = pd.DataFrame(candidate_columns)

numeric_for_signal = df_integrated_major[["평균소득만원", "중위소득만원", *feature_cols_primary]].copy()
pearson = numeric_for_signal.corr(method="pearson", numeric_only=True)["평균소득만원"].drop("평균소득만원")
spearman = numeric_for_signal.corr(method="spearman", numeric_only=True)["평균소득만원"].drop("평균소득만원")
signal_rank = (
    pd.DataFrame({"pearson_vs_평균소득": pearson, "spearman_vs_평균소득": spearman})
    .assign(abs_spearman=lambda x: x["spearman_vs_평균소득"].abs())
    .sort_values("abs_spearman", ascending=False)
)

major_rank = df_integrated_major[["계열", "평균소득만원", "중위소득만원", "대중견기업비율", "공공비영리비율", "자격증 취득률", "상위산업", "상위산업비율"]].sort_values("평균소득만원", ascending=False)

insight_rows = []
top_income = major_rank.iloc[0]
low_income = major_rank.iloc[-1]
insight_rows.append({
    "관찰": "평균소득 상위/하위 계열 격차가 크다.",
    "근거": f"{top_income['계열']} {top_income['평균소득만원']:.1f}만원 vs {low_income['계열']} {low_income['평균소득만원']:.1f}만원",
    "해석": "통합 샘플의 1차 타겟은 평균소득만원으로 두고, 중위소득만원으로 평균의 쏠림을 점검한다.",
})
if "중소기업_비율" in signal_rank.index:
    insight_rows.append({
        "관찰": "중소기업 비율은 평균소득과 음의 방향 신호가 나온다.",
        "근거": f"Spearman {signal_rank.loc['중소기업_비율', 'spearman_vs_평균소득']:.3f}, Pearson {signal_rank.loc['중소기업_비율', 'pearson_vs_평균소득']:.3f}",
        "해석": "계열별 취업처 규모 분포가 급여 차이를 설명하는 후보 피처다. 단, 표본은 7계열뿐이라 가설 수준이다.",
    })
if "자격증 취득률" in signal_rank.index:
    insight_rows.append({
        "관찰": "자격증 취득률은 단독으로 고소득을 설명하기 어렵다.",
        "근거": f"Spearman {signal_rank.loc['자격증 취득률', 'spearman_vs_평균소득']:.3f}, 의약은 자격증 취득률이 낮지만 평균소득은 높다.",
        "해석": "자격증은 계열별 노동시장 구조와 함께 보아야 한다.",
    })
insight_rows.append({
    "관찰": "산업 집중도는 계열 성격을 잘 드러내지만 곧바로 인과로 읽으면 위험하다.",
    "근거": "의약은 보건·사회복지업 집중, 공학은 제조업 집중, 교육은 교육서비스업 집중이 뚜렷하다.",
    "해석": "산업 믹스는 설명 피처로 적합하지만 학교/학과 수준 표본이 없어 계열 평균 비교에 머문다.",
})
insight_rows.append({
    "관찰": "직무별 자격증 자료는 현재 통합 키가 없다.",
    "근거": "df_cert_job의 키는 직무분류이고 나머지 주요 표는 계열이다.",
    "해석": "직무-계열 또는 직무-산업 매핑표가 생기면 자격증 직무 집중도를 계열 피처로 확장할 수 있다.",
})
df_integrated_insights = pd.DataFrame(insight_rows)

display(Markdown("## 8. 후보 컬럼·타겟 설계와 통합 샘플 EDA"))
display(Markdown("### 타겟 후보"))
display(target_candidates)
display(Markdown("### 후보 컬럼 목록"))
display(candidate_columns)
display(Markdown("### 평균소득만원 기준 계열 순위"))
display(major_rank)
display(Markdown("### 후보 피처와 평균소득만원의 상관 신호: n=7이므로 가설 탐색용"))
display(signal_rank)
display(Markdown("### 통합 샘플에서 나오는 1차 인사이트"))
display(df_integrated_insights)


## 8. 후보 컬럼·타겟 설계와 통합 샘플 EDA

### 타겟 후보

,target_column,role,why,caveat
0,평균소득만원,primary_y,연속형이고 계열별 초임급여 수준 차이를 직접 보여준다.,n=7이면 모델링보다 가설 탐색용이다. 표본 확장은 학과/학교 단위 원자료가 필요하다.
1,log10_평균소득만원,primary_y_transformed_when_modeling,소득 타겟은 양수이고 상위 계열 영향이 커질 수 있어 곱셈적 차이를 완화한다.,현재 n=7에서는 변환 전후 순위가 거의 같으므로 설명용 보조 컬럼이다.
2,중위소득만원,secondary_y,평균이 일부 고소득자에 끌릴 때 전형적 졸업자 소득을 보완한다.,평균소득과 함께 보아야 하며 단독 타겟으로 쓰면 고소득 꼬리 정보를 잃는다.
3,300만원이상_비율 / 400만원이상_비율,distribution_target_or_segment_y,초임급여의 상위 구간 진입률을 설명할 때 기사형 타겟으로 해석하기 쉽다.,평균소득의 설명변수로 동시에 넣으면 타겟 누수에 가깝다. 별도 타겟으로만 사용한다.
4,자격증 취득률,alternate_y,취업성과가 아니라 계열별 자격 취득 행동 자체를 설명하려는 경우 타겟이 될 수 있다.,초임급여 EDA의 설명변수로 쓸 때와 역할을 분리한다.


### 후보 컬럼 목록

,column,role,source,use
0,평균소득만원,target_or_target_diagnostic,df_salary,타겟 설계/급여 분포 진단
1,중위소득만원,target_or_target_diagnostic,df_salary,타겟 설계/급여 분포 진단
2,log10_평균소득만원,target_or_target_diagnostic,df_salary,타겟 설계/급여 분포 진단
3,300만원이상_비율,target_or_target_diagnostic,df_salary,타겟 설계/급여 분포 진단
4,400만원이상_비율,target_or_target_diagnostic,df_salary,타겟 설계/급여 분포 진단
5,대기업_비율,feature,df_company,계열별 취업처 규모/공공성 구조
6,중견기업_비율,feature,df_company,계열별 취업처 규모/공공성 구조
7,중소기업_비율,feature,df_company,계열별 취업처 규모/공공성 구조
8,대중견기업비율,feature,df_company,계열별 취업처 규모/공공성 구조
9,공공비영리비율,feature,df_company,계열별 취업처 규모/공공성 구조


...

,계열,평균소득만원,중위소득만원,대중견기업비율,공공비영리비율,자격증 취득률,상위산업,상위산업비율
1,사회,364.300,270.700,18.453,35.060,45.700,Q.보건·사회복지업,13.178
3,공학,361.700,308.300,33.485,16.060,61.000,C.제조업,38.123
5,의약,360.200,294.400,2.742,50.343,27.700,Q.보건·사회복지업,77.280
2,교육,329.700,290.600,2.918,61.120,37.800,P.교육서비스업,43.528
0,인문,316.300,255.900,16.578,38.627,42.900,P.교육서비스업,13.350
4,자연,311.000,260.000,21.608,26.102,58.300,C.제조업,24.965
6,예체능,245.100,220.000,12.382,17.327,33.200,G.도·소매업,13.660


### 후보 피처와 평균소득만원의 상관 신호: n=7이므로 가설 탐색용

,pearson_vs_평균소득,spearman_vs_평균소득,abs_spearman
중위소득만원,0.879,0.750,0.750
예술스포츠여가비율,-0.931,-0.750,0.750
공공기관_비율,0.712,0.643,0.643
중소기업_비율,-0.489,-0.571,0.571
상위3산업비율,0.496,0.464,0.464
1인당 자격증 취득수,0.488,0.414,0.414
산업허핀달지수,0.439,0.393,0.393
교육서비스비율,-0.140,-0.357,0.357
공공행정비율,0.201,0.286,0.286
자격증 취득률,0.242,0.286,0.286


### 통합 샘플에서 나오는 1차 인사이트

,관찰,근거,해석
0,평균소득 상위/하위 계열 격차가 크다.,사회 364.3만원 vs 예체능 245.1만원,"통합 샘플의 1차 타겟은 평균소득만원으로 두고, 중위소득만원으로 평균의 쏠림을 점검한다."
1,중소기업 비율은 평균소득과 음의 방향 신호가 나온다.,"Spearman -0.571, Pearson -0.489","계열별 취업처 규모 분포가 급여 차이를 설명하는 후보 피처다. 단, 표본은 7계열뿐..."
2,자격증 취득률은 단독으로 고소득을 설명하기 어렵다.,"Spearman 0.286, 의약은 자격증 취득률이 낮지만 평균소득은 높다.",자격증은 계열별 노동시장 구조와 함께 보아야 한다.
3,산업 집중도는 계열 성격을 잘 드러내지만 곧바로 인과로 읽으면 위험하다.,"의약은 보건·사회복지업 집중, 공학은 제조업 집중, 교육은 교육서비스업 집중이 뚜렷하다.",산업 믹스는 설명 피처로 적합하지만 학교/학과 수준 표본이 없어 계열 평균 비교에 ...
4,직무별 자격증 자료는 현재 통합 키가 없다.,df_cert_job의 키는 직무분류이고 나머지 주요 표는 계열이다.,직무-계열 또는 직무-산업 매핑표가 생기면 자격증 직무 집중도를 계열 피처로 확장할...


In [10]:

plot_df = df_integrated_major.sort_values("평균소득만원", ascending=False).copy()

fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
fig.suptitle("통합 샘플 EDA: 계열별 초임급여 타겟과 후보 피처", fontsize=15)

ax = axes[0, 0]
ax.bar(plot_df["계열"], plot_df["평균소득만원"], color="#4C78A8", label="평균소득")
ax.plot(plot_df["계열"], plot_df["중위소득만원"], color="#F58518", marker="o", linewidth=2, label="중위소득")
ax.set_title("평균/중위 초임급여")
ax.set_ylabel("만원")
ax.legend(loc="upper right")

ax = axes[0, 1]
width = 0.36
x = np.arange(len(plot_df))
ax.bar(x - width / 2, plot_df["대중견기업비율"], width=width, color="#54A24B", label="대+중견기업")
ax.bar(x + width / 2, plot_df["공공비영리비율"], width=width, color="#B279A2", label="공공+비영리")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["계열"])
ax.set_title("취업처 구조 후보 피처")
ax.set_ylabel("비율(%)")
ax.legend(loc="upper right")

ax = axes[1, 0]
top_signal = signal_rank.drop(index=["중위소득만원"], errors="ignore").head(10).sort_values("spearman_vs_평균소득")
colors = np.where(top_signal["spearman_vs_평균소득"] >= 0, "#72B7B2", "#E45756")
ax.barh(top_signal.index, top_signal["spearman_vs_평균소득"], color=colors)
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set_title("평균소득만원과 Spearman 상관 Top 10")
ax.set_xlabel("Spearman rho")

ax = axes[1, 1]
industry_cols = ["제조업비율", "정보통신업비율", "전문과학기술비율", "교육서비스비율", "보건사회복지비율"]
heat = df_integrated_major.set_index("계열")[industry_cols].loc[plot_df["계열"]]
im = ax.imshow(heat.values, aspect="auto", cmap="YlGnBu")
ax.set_xticks(np.arange(len(industry_cols)))
ax.set_xticklabels([c.replace("비율", "") for c in industry_cols], rotation=35, ha="right")
ax.set_yticks(np.arange(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_title("주요 산업 비율 히트맵")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.iloc[i, j]:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="비율(%)")

display(fig)
plt.close(fig)


<Figure size 1500x1000 with 5 Axes>

In [11]:

recommended_feature_set = pd.DataFrame(
    [
        {"priority": 1, "column": "대중견기업비율", "type": "numeric_feature", "reason": "고임금 가능성이 큰 취업처 규모 구조"},
        {"priority": 2, "column": "중소기업_비율", "type": "numeric_feature", "reason": "평균소득과 반대 방향 신호가 관찰됨"},
        {"priority": 3, "column": "공공비영리비율", "type": "numeric_feature", "reason": "교육/의약 계열처럼 공공·비영리 비중이 큰 집단 분리"},
        {"priority": 4, "column": "자격증 취득률", "type": "numeric_feature", "reason": "계열별 자격 취득 참여율"},
        {"priority": 5, "column": "1인당 자격증 취득수", "type": "numeric_feature", "reason": "취득자 1인당 자격증 강도"},
        {"priority": 6, "column": "상위3산업비율", "type": "numeric_feature", "reason": "계열별 취업 산업 집중도"},
        {"priority": 7, "column": "제조업비율/보건사회복지비율/교육서비스비율/전문과학기술비율", "type": "numeric_feature_group", "reason": "계열별 대표 산업 믹스"},
        {"priority": 8, "column": "상위산업", "type": "categorical_context", "reason": "표본이 커질 때 범주형 설명 변수 또는 라벨로 사용"},
    ]
)

final_decision = f"""
## 9. 결론: 통합 샘플 설계 결정

**Target 설계**

- 1순위 타겟: `평균소득만원`
- 보조 타겟: `중위소득만원`
- 모델링용 변환 후보: `log10_평균소득만원`
- 별도 기사형 타겟 후보: `300만원이상_비율`, `400만원이상_비율`

**추천 통합 샘플**

- `df_integrated_major`: `고등교육기관 x 계열` 7행.
- 포함 소스: 기업유형, 계열별 자격증, 초임급여, 산업유형.
- 제외/보조 소스: `df_cert_job`은 `직무분류` 축이라 직접 조인하지 않는다.

**관찰**

- 평균소득만원 상위는 `{top_income['계열']}`({top_income['평균소득만원']:.1f}만원), 하위는 `{low_income['계열']}`({low_income['평균소득만원']:.1f}만원)이다.
- 취업처 구조, 자격증 취득 강도, 산업 믹스가 계열별 소득 차이를 설명하는 후보 피처가 된다.
- 다만 `df_integrated_major`는 7행뿐이므로 통계 모델 결론이 아니라 EDA 가설 설계 단계로 제한한다.

**다음 확장 방향**

- 학부/대학원 구분까지 늘릴 때는 `df_integrated_degree_major` 21행을 쓴다. 이 경우 산업유형은 빠진다.
- 직무별 자격증을 통합하려면 `직무분류 -> 계열` 또는 `직무분류 -> 산업분류` 매핑표가 추가로 필요하다.
- 더 강한 분석은 학교/학과 단위 원자료와 이 계열별 지표를 붙여 표본 수를 늘린 뒤 수행해야 한다.
"""

display(Markdown("### 추천 피처 세트"))
display(recommended_feature_set)
display(Markdown(final_decision))


### 추천 피처 세트

,priority,column,type,reason
0,1,대중견기업비율,numeric_feature,고임금 가능성이 큰 취업처 규모 구조
1,2,중소기업_비율,numeric_feature,평균소득과 반대 방향 신호가 관찰됨
2,3,공공비영리비율,numeric_feature,교육/의약 계열처럼 공공·비영리 비중이 큰 집단 분리
3,4,자격증 취득률,numeric_feature,계열별 자격 취득 참여율
4,5,1인당 자격증 취득수,numeric_feature,취득자 1인당 자격증 강도
5,6,상위3산업비율,numeric_feature,계열별 취업 산업 집중도
6,7,제조업비율/보건사회복지비율/교육서비스비율/전문과학기술비율,numeric_feature_group,계열별 대표 산업 믹스
7,8,상위산업,categorical_context,표본이 커질 때 범주형 설명 변수 또는 라벨로 사용



## 9. 결론: 통합 샘플 설계 결정

**Target 설계**

- 1순위 타겟: `평균소득만원`
- 보조 타겟: `중위소득만원`
- 모델링용 변환 후보: `log10_평균소득만원`
- 별도 기사형 타겟 후보: `300만원이상_비율`, `400만원이상_비율`

**추천 통합 샘플**

- `df_integrated_major`: `고등교육기관 x 계열` 7행.
- 포함 소스: 기업유형, 계열별 자격증, 초임급여, 산업유형.
- 제외/보조 소스: `df_cert_job`은 `직무분류` 축이라 직접 조인하지 않는다.

**관찰**

- 평균소득만원 상위는 `사회`(364.3만원), 하위는 `예체능`(245.1만원)이다.
- 취업처 구조, 자격증 취득 강도, 산업 믹스가 계열별 소득 차이를 설명하는 후보 피처가 된다.
- 다만 `df_integrated_major`는 7행뿐이므로 통계 모델 결론이 아니라 EDA 가설 설계 단계로 제한한다.

**다음 확장 방향**

- 학부/대학원 구분까지 늘릴 때는 `df_integrated_degree_major` 21행을 쓴다. 이 경우 산업유형은 빠진다.
- 직무별 자격증을 통합하려면 `직무분류 -> 계열` 또는 `직무분류 -> 산업분류` 매핑표가 추가로 필요하다.
- 더 강한 분석은 학교/학과 단위 원자료와 이 계열별 지표를 붙여 표본 수를 늘린 뒤 수행해야 한다.


## 10. 데이터 핸들링/클리닝 점검: 0, NULL, 원천 코드형 결측 분리

이 블록은 각 샘플의 컬럼 의미를 먼저 분류한 뒤 결측을 다시 계산한다.

- **경쟁형 구성비 컬럼**: 한 행 안에서 여러 선택지가 같은 분모를 나눠 갖는 컬럼이다. 예: 기업유형 7개, 초임급여 구간 5개, 산업분류별 계열 수, 직무별 자격증 수. 이때 `0`은 관측된 0명/0건이므로 결측이 아니다.
- **단순 범주형 컬럼**: 키나 라벨이다. 예: `구분`, `계열`, `산업분류`, `직무분류`. 이 컬럼의 `NULL`은 조인/집계 단위를 깨뜨리는 실질 결측이다.
- **원천 코드형 미상/결측값**: 문자열 값 자체가 `결측값`, `미상`인 경우다. pandas `NULL`은 아니지만, 기사/모델 해석에서는 실질적 분류 미상으로 따로 잡는다.
- **파생비율 NULL**: 분모가 `0` 또는 `NULL`이라 계산할 수 없는 경우다. 원자료 결측과 구분해 구조적 NULL로 표시한다.


In [12]:

SEMANTIC_RULES = {
    "df_company": {
        "df": df_company,
        "unit": "구분 x 계열",
        "key_cols": ["구분", "계열"],
        "denominator": "분석대상자",
        "competitive_groups": {
            "기업유형_취업자수": ["대기업", "중견기업", "중소기업", "국가및지방", "공공기관", "비영리", "기타"],
            "기업유형_비율": ["대기업_비율", "중견기업_비율", "중소기업_비율", "국가및지방_비율", "공공기관_비율", "비영리_비율", "기타_비율"],
        },
        "simple_categories": ["구분", "계열"],
        "target_cols": [],
        "source_unknown_values": {},
        "note": "기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.",
    },
    "df_cert_major": {
        "df": df_cert_major,
        "unit": "구분 x 계열",
        "key_cols": ["구분", "계열"],
        "denominator": "분석대상자",
        "competitive_groups": {},
        "simple_categories": ["구분", "계열"],
        "target_cols": ["자격증 취득률"],
        "source_unknown_values": {},
        "note": "자격증취득자는 취득/미취득의 이항 구조이고, 자격증취득수는 취득자의 복수 취득 강도다.",
    },
    "df_salary": {
        "df": df_salary,
        "unit": "구분 x 계열",
        "key_cols": ["구분", "계열"],
        "denominator": "분석대상자",
        "competitive_groups": {
            "초임급여구간_취업자수": ["100만원미만", "100~200만원미만", "200~300만원미만", "300~400만원미만", "400만원이상"],
            "초임급여구간_비율": ["100만원미만_비율", "100~200만원미만_비율", "200~300만원미만_비율", "300~400만원미만_비율", "400만원이상_비율"],
        },
        "simple_categories": ["구분", "계열"],
        "target_cols": ["평균소득만원", "중위소득만원", "300만원이상_비율", "400만원이상_비율"],
        "source_unknown_values": {},
        "note": "초임급여 구간은 상호배타적 경쟁형 구간이고 평균/중위소득은 연속 타겟 후보다.",
    },
    "df_industry": {
        "df": df_industry,
        "unit": "산업분류 x 계열(wide)",
        "key_cols": ["산업분류"],
        "denominator": "분석대상자",
        "competitive_groups": {
            "계열별_산업취업자수": [col for col in MAJORS if col in df_industry.columns],
        },
        "simple_categories": ["산업분류"],
        "target_cols": [],
        "source_unknown_values": {"산업분류": ["결측값"]},
        "note": "wide 행 기준으로 계열 컬럼 합은 산업분류별 전체와 경쟁형으로 맞아야 한다. 단, 결측값 행은 산업분류 미상 규모다.",
    },
    "df_industry_long": {
        "df": df_industry_long,
        "unit": "산업분류 x 계열(long)",
        "key_cols": ["산업분류", "계열"],
        "denominator": "계열전체취업자수",
        "competitive_groups": {},
        "simple_categories": ["산업분류", "계열"],
        "target_cols": [],
        "source_unknown_values": {"산업분류": ["결측값"]},
        "note": "long 기준에서 산업분류는 계열 내부 취업처 구성을 나누는 경쟁형 범주다.",
    },
    "df_cert_job": {
        "df": df_cert_job,
        "unit": "직무분류",
        "key_cols": ["직무분류"],
        "denominator": "고등교육기관",
        "competitive_groups": {
            "학위구분별_자격증취득수": ["학부", "대학원"],
        },
        "simple_categories": ["직무분류"],
        "target_cols": [],
        "source_unknown_values": {"직무분류": ["미상"]},
        "note": "직무분류는 자격증 취득수를 직무 분야로 나누는 경쟁형 범주다. 미상은 원천 분류 미상이다.",
    },
    "df_cert_job_long": {
        "df": df_cert_job_long,
        "unit": "직무분류 x 구분",
        "key_cols": ["직무분류", "구분"],
        "denominator": None,
        "competitive_groups": {},
        "simple_categories": ["직무분류", "구분"],
        "target_cols": [],
        "source_unknown_values": {"직무분류": ["미상"]},
        "note": "long 기준에서 구분은 단순 범주이고 직무분류 미상은 pandas NULL과 별도로 관리한다.",
    },
    "df_integrated_major": {
        "df": df_integrated_major,
        "unit": "고등교육기관 x 계열",
        "key_cols": ["계열"],
        "denominator": "분석대상자",
        "competitive_groups": {},
        "simple_categories": ["계열", "상위산업"],
        "target_cols": ["평균소득만원", "중위소득만원", "log10_평균소득만원", "300만원이상_비율", "400만원이상_비율"],
        "source_unknown_values": {},
        "note": "4개 계열축 자료를 결합한 최종 1차 샘플이다. merge 후 NULL은 조인 실패로 본다.",
    },
    "df_integrated_degree_major": {
        "df": df_integrated_degree_major,
        "unit": "구분 x 계열",
        "key_cols": ["구분", "계열"],
        "denominator": "분석대상자",
        "competitive_groups": {},
        "simple_categories": ["구분", "계열"],
        "target_cols": ["평균소득만원", "중위소득만원", "log10_평균소득만원", "300만원이상_비율"],
        "source_unknown_values": {},
        "note": "산업유형을 제외하고 학위 구분까지 확장한 21행 후보 샘플이다.",
    },
}

semantic_rows = []
for dataset, spec in SEMANTIC_RULES.items():
    df_ = spec["df"]
    competitive_cols = {col: group for group, cols in spec["competitive_groups"].items() for col in cols if col in df_.columns}
    for col in df_.columns:
        if col in spec["key_cols"]:
            role = "join_key"
            semantic_type = "simple_category"
            zero_policy = "zero_not_expected"
            null_policy = "actual_missing_if_null"
        elif col in spec["simple_categories"]:
            role = "category_context"
            semantic_type = "simple_category"
            zero_policy = "zero_not_expected"
            null_policy = "actual_missing_if_null"
        elif col in competitive_cols:
            role = "composition_component"
            semantic_type = "competitive_count_or_share"
            zero_policy = "observed_zero_keep"
            null_policy = "actual_missing_unless_structural_denominator_null"
        elif col in spec["target_cols"]:
            role = "target_or_target_diagnostic"
            semantic_type = "continuous_or_rate_target"
            zero_policy = "valid_only_if_domain_allows_zero"
            null_policy = "actual_missing_for_target"
        elif pd.api.types.is_numeric_dtype(df_[col]):
            role = "numeric_measure_or_engineered_feature"
            semantic_type = "numeric_measure"
            zero_policy = "observed_zero_keep_after_domain_check"
            null_policy = "actual_missing_or_structural_if_denominator_null"
        else:
            role = "category_context"
            semantic_type = "simple_category"
            zero_policy = "zero_not_expected"
            null_policy = "actual_missing_if_null"
        semantic_rows.append(
            {
                "dataset": dataset,
                "unit": spec["unit"],
                "column": col,
                "role": role,
                "semantic_type": semantic_type,
                "competitive_group": competitive_cols.get(col, ""),
                "zero_policy": zero_policy,
                "null_policy": null_policy,
                "dataset_note": spec["note"],
            }
        )

df_column_semantics = pd.DataFrame(semantic_rows)

display(Markdown("### 10-1. 컬럼 의미 분류표"))
display(df_column_semantics)
display(Markdown("### 경쟁형 구성 컬럼만 보기"))
display(df_column_semantics[df_column_semantics["semantic_type"] == "competitive_count_or_share"])


### 10-1. 컬럼 의미 분류표

,dataset,unit,column,role,semantic_type,competitive_group,zero_policy,null_policy,dataset_note
0,df_company,구분 x 계열,구분,join_key,simple_category,,zero_not_expected,actual_missing_if_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
1,df_company,구분 x 계열,계열,join_key,simple_category,,zero_not_expected,actual_missing_if_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
2,df_company,구분 x 계열,분석대상자,numeric_measure_or_engineered_feature,numeric_measure,,observed_zero_keep_after_domain_check,actual_missing_or_structural_if_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
3,df_company,구분 x 계열,대기업,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
4,df_company,구분 x 계열,중견기업,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
...,...,...,...,...,...,...,...,...,...
120,df_integrated_degree_major,구분 x 계열,공공비영리비율,numeric_measure_or_engineered_feature,numeric_measure,,observed_zero_keep_after_domain_check,actual_missing_or_structural_if_denominator_null,산업유형을 제외하고 학위 구분까지 확장한 21행 후보 샘플이다.
121,df_integrated_degree_major,구분 x 계열,자격증 취득률,numeric_measure_or_engineered_feature,numeric_measure,,observed_zero_keep_after_domain_check,actual_missing_or_structural_if_denominator_null,산업유형을 제외하고 학위 구분까지 확장한 21행 후보 샘플이다.
122,df_integrated_degree_major,구분 x 계열,자격증취득수,numeric_measure_or_engineered_feature,numeric_measure,,observed_zero_keep_after_domain_check,actual_missing_or_structural_if_denominator_null,산업유형을 제외하고 학위 구분까지 확장한 21행 후보 샘플이다.
123,df_integrated_degree_major,구분 x 계열,1인당 자격증 취득수,numeric_measure_or_engineered_feature,numeric_measure,,observed_zero_keep_after_domain_check,actual_missing_or_structural_if_denominator_null,산업유형을 제외하고 학위 구분까지 확장한 21행 후보 샘플이다.


### 경쟁형 구성 컬럼만 보기

,dataset,unit,column,role,semantic_type,competitive_group,zero_policy,null_policy,dataset_note
3,df_company,구분 x 계열,대기업,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
4,df_company,구분 x 계열,중견기업,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
5,df_company,구분 x 계열,중소기업,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
6,df_company,구분 x 계열,국가및지방,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
7,df_company,구분 x 계열,공공기관,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
8,df_company,구분 x 계열,비영리,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
9,df_company,구분 x 계열,기타,composition_component,competitive_count_or_share,기업유형_취업자수,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
10,df_company,구분 x 계열,대기업_비율,composition_component,competitive_count_or_share,기업유형_비율,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
11,df_company,구분 x 계열,중견기업_비율,composition_component,competitive_count_or_share,기업유형_비율,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.
12,df_company,구분 x 계열,중소기업_비율,composition_component,competitive_count_or_share,기업유형_비율,observed_zero_keep,actual_missing_unless_structural_denominator_null,기업유형은 같은 취업자를 7개 유형으로 나누는 경쟁형 구성이다.


In [13]:

def numeric_zero_count(series: pd.Series) -> int:
    if not pd.api.types.is_numeric_dtype(series):
        return 0
    return int((series.fillna(np.nan) == 0).sum())


def source_unknown_cell_count(df_: pd.DataFrame, source_unknown_values: dict[str, list[str]], col: str) -> int:
    values = source_unknown_values.get(col, [])
    if not values or col not in df_.columns:
        return 0
    return int(df_[col].astype("string").isin(values).sum())


def structural_null_count(df_: pd.DataFrame, col: str, denominator: str | None) -> int:
    if denominator is None or denominator not in df_.columns or col not in df_.columns:
        return 0
    if not pd.api.types.is_numeric_dtype(df_[col]):
        return 0
    denom = pd.to_numeric(df_[denominator], errors="coerce")
    return int(df_[col].isna().sum() and ((denom.isna()) | (denom == 0)).sum())


missing_rows = []
category_rows = []
source_unknown_rows = []
for dataset, spec in SEMANTIC_RULES.items():
    df_ = spec["df"]
    semantics = df_column_semantics[df_column_semantics["dataset"] == dataset].set_index("column")
    for col in df_.columns:
        semantic_type = semantics.loc[col, "semantic_type"] if col in semantics.index else "unknown"
        source_unknown_n = source_unknown_cell_count(df_, spec["source_unknown_values"], col)
        structural_n = structural_null_count(df_, col, spec["denominator"])
        pandas_null_n = int(df_[col].isna().sum())
        actual_missing_n = max(pandas_null_n - structural_n, 0) + source_unknown_n
        missing_rows.append(
            {
                "dataset": dataset,
                "column": col,
                "semantic_type": semantic_type,
                "rows": len(df_),
                "pandas_null_n": pandas_null_n,
                "source_coded_unknown_n": source_unknown_n,
                "structural_null_n": structural_n,
                "actual_missing_n": actual_missing_n,
                "actual_missing_pct": actual_missing_n / len(df_) * 100 if len(df_) else np.nan,
                "zero_n": numeric_zero_count(df_[col]),
                "zero_pct": numeric_zero_count(df_[col]) / len(df_) * 100 if len(df_) else np.nan,
                "unique_n": int(df_[col].nunique(dropna=True)),
            }
        )
        if semantic_type == "simple_category":
            top_values = df_[col].astype("string").value_counts(dropna=False).head(8)
            category_rows.append(
                {
                    "dataset": dataset,
                    "column": col,
                    "unique_n": int(df_[col].nunique(dropna=True)),
                    "top_values": " | ".join([f"{idx}:{val}" for idx, val in top_values.items()]),
                    "source_unknown_values": ", ".join(spec["source_unknown_values"].get(col, [])),
                }
            )
    for col, values in spec["source_unknown_values"].items():
        if col not in df_.columns:
            continue
        for value in values:
            mask = df_[col].astype("string").eq(value)
            numeric_cols = df_.select_dtypes(include="number").columns.tolist()
            source_unknown_rows.append(
                {
                    "dataset": dataset,
                    "column": col,
                    "source_unknown_value": value,
                    "rows_with_value": int(mask.sum()),
                    "numeric_sum_snapshot": {num_col: float(df_.loc[mask, num_col].sum()) for num_col in numeric_cols[:8]},
                    "note": "문자열 값은 NULL이 아니지만 분류 미상 규모이므로 실질 결측성 지표로 별도 관리",
                }
            )

df_zero_null_profile = pd.DataFrame(missing_rows).sort_values(["actual_missing_n", "pandas_null_n", "zero_n"], ascending=[False, False, False])
df_category_profile = pd.DataFrame(category_rows)
df_source_coded_unknown = pd.DataFrame(source_unknown_rows)

real_missing_summary = (
    df_zero_null_profile.groupby("dataset", as_index=False)
    .agg(
        rows_observed=("rows", "max"),
        columns=("column", "nunique"),
        pandas_null_cells=("pandas_null_n", "sum"),
        source_coded_unknown_cells=("source_coded_unknown_n", "sum"),
        structural_null_cells=("structural_null_n", "sum"),
        actual_missing_cells=("actual_missing_n", "sum"),
        zero_cells=("zero_n", "sum"),
    )
)
real_missing_summary["handling_decision"] = np.where(
    real_missing_summary["actual_missing_cells"].eq(0),
    "실질 결측 없음. 0은 관측값으로 유지",
    "실질 결측/원천 미상 값 별도 플래그 필요",
)

unknown_volume_rows = []
if "산업분류" in df_industry.columns and "결측값" in set(df_industry["산업분류"].astype("string")):
    industry_unknown = df_industry[df_industry["산업분류"].astype("string").eq("결측값")]
    industry_total = df_industry[df_industry["산업분류"].astype("string").eq("전체")]
    for major in MAJORS:
        unknown_value = float(industry_unknown[major].sum()) if major in industry_unknown else 0.0
        total_value = float(industry_total[major].sum()) if major in industry_total else np.nan
        unknown_volume_rows.append(
            {
                "dataset": "df_industry",
                "dimension": major,
                "unknown_label": "산업분류=결측값",
                "unknown_count": unknown_value,
                "denominator": total_value,
                "unknown_pct": unknown_value / total_value * 100 if total_value else np.nan,
            }
        )
if "직무분류" in df_cert_job.columns and "미상" in set(df_cert_job["직무분류"].astype("string")):
    cert_job_unknown = df_cert_job[df_cert_job["직무분류"].astype("string").eq("미상")]
    cert_job_total_row = df_cert_job[df_cert_job["직무분류"].astype("string").eq("전체")]
    for group_col in ["고등교육기관", "학부", "대학원"]:
        unknown_value = float(cert_job_unknown[group_col].sum())
        total_value = float(cert_job_total_row[group_col].sum())
        unknown_volume_rows.append(
            {
                "dataset": "df_cert_job",
                "dimension": group_col,
                "unknown_label": "직무분류=미상",
                "unknown_count": unknown_value,
                "denominator": total_value,
                "unknown_pct": unknown_value / total_value * 100 if total_value else np.nan,
            }
        )
df_source_unknown_volume = pd.DataFrame(unknown_volume_rows)

display(Markdown("### 10-2. 0/NULL/원천 코드형 미상 분리 요약"))
display(real_missing_summary)
display(Markdown("### 컬럼별 0/NULL/실질 결측 프로파일"))
display(df_zero_null_profile)
display(Markdown("### 단순 범주형 프로파일"))
display(df_category_profile)
display(Markdown("### 원천 코드형 미상/결측값 규모"))
display(df_source_unknown_volume if len(df_source_unknown_volume) else pd.DataFrame([{"status": "원천 코드형 미상/결측값 없음"}]))


### 10-2. 0/NULL/원천 코드형 미상 분리 요약

,dataset,rows_observed,columns,pandas_null_cells,source_coded_unknown_cells,structural_null_cells,actual_missing_cells,zero_cells,handling_decision
0,df_cert_job,24,5,0,1,0,1,0,실질 결측/원천 미상 값 별도 플래그 필요
1,df_cert_job_long,72,3,0,3,0,3,0,실질 결측/원천 미상 값 별도 플래그 필요
2,df_cert_major,24,9,0,0,0,0,0,실질 결측 없음. 0은 관측값으로 유지
3,df_company,24,19,0,0,0,0,0,실질 결측 없음. 0은 관측값으로 유지
4,df_industry,22,9,0,1,0,1,0,실질 결측/원천 미상 값 별도 플래그 필요
5,df_industry_long,154,6,0,7,0,7,0,실질 결측/원천 미상 값 별도 플래그 필요
6,df_integrated_degree_major,21,16,0,0,0,0,0,실질 결측 없음. 0은 관측값으로 유지
7,df_integrated_major,7,39,0,0,0,0,0,실질 결측 없음. 0은 관측값으로 유지
8,df_salary,40,19,0,0,0,0,0,실질 결측 없음. 0은 관측값으로 유지


### 컬럼별 0/NULL/실질 결측 프로파일

,dataset,column,semantic_type,rows,pandas_null_n,source_coded_unknown_n,structural_null_n,actual_missing_n,actual_missing_pct,zero_n,zero_pct,unique_n
56,df_industry_long,산업분류,simple_category,154,0,7,0,7,4.545,0,0.000,22
67,df_cert_job_long,직무분류,simple_category,72,0,3,0,3,4.167,0,0.000,24
47,df_industry,산업분류,simple_category,22,0,1,0,1,4.545,0,0.000,22
62,df_cert_job,직무분류,simple_category,24,0,1,0,1,4.167,0,0.000,24
0,df_company,구분,simple_category,24,0,0,0,0,0.000,0,0.000,3
...,...,...,...,...,...,...,...,...,...,...,...,...
120,df_integrated_degree_major,공공비영리비율,numeric_measure,21,0,0,0,0,0.000,0,0.000,21
121,df_integrated_degree_major,자격증 취득률,numeric_measure,21,0,0,0,0,0.000,0,0.000,21
122,df_integrated_degree_major,자격증취득수,numeric_measure,21,0,0,0,0,0.000,0,0.000,21
123,df_integrated_degree_major,1인당 자격증 취득수,numeric_measure,21,0,0,0,0,0.000,0,0.000,10


### 단순 범주형 프로파일

,dataset,column,unique_n,top_values,source_unknown_values
0,df_company,구분,3,고등교육기관:8 | 학부:8 | 대학원:8,
1,df_company,계열,8,전체:3 | 인문:3 | 사회:3 | 교육:3 | 공학:3 | 자연:3 | 의약:3...,
2,df_cert_major,구분,3,고등교육기관:8 | 학부:8 | 대학원:8,
3,df_cert_major,계열,8,전체:3 | 인문:3 | 사회:3 | 교육:3 | 공학:3 | 자연:3 | 의약:3...,
4,df_salary,구분,5,고등교육기관:8 | 학부:8 | 대학원:8 | (석사):8 | (박사):8,
5,df_salary,계열,9,인문:5 | 사회:5 | 교육:5 | 공학:5 | 자연:5 | 의약:5 | 예체능:...,
6,df_industry,산업분류,22,전체:1 | A.농림어업:1 | B.광업:1 | C.제조업:1 | D.전기·가스·증...,결측값
7,df_industry_long,산업분류,22,전체:7 | A.농림어업:7 | B.광업:7 | C.제조업:7 | D.전기·가스·증...,결측값
8,df_industry_long,계열,7,인문:22 | 사회:22 | 교육:22 | 공학:22 | 자연:22 | 의약:22 ...,
9,df_cert_job,직무분류,24,전체:1 | 경영·회계·사무:1 | 교육·자연과학·사회과학:1 | 보건·의료:1 |...,미상


### 원천 코드형 미상/결측값 규모

,dataset,dimension,unknown_label,unknown_count,denominator,unknown_pct
0,df_industry,인문,산업분류=결측값,656.000,"23,543.000",2.786
1,df_industry,사회,산업분류=결측값,"1,723.000","82,516.000",2.088
2,df_industry,교육,산업분류=결측값,"2,605.000","25,602.000",10.175
3,df_industry,공학,산업분류=결측값,813.000,"86,625.000",0.939
4,df_industry,자연,산업분류=결측값,458.000,"28,212.000",1.623
5,df_industry,의약,산업분류=결측값,373.000,"47,637.000",0.783
6,df_industry,예체능,산업분류=결측값,667.000,"25,117.000",2.656
7,df_cert_job,고등교육기관,직무분류=미상,"2,313.000","303,447.000",0.762
8,df_cert_job,학부,직무분류=미상,"1,918.000","233,700.000",0.821
9,df_cert_job,대학원,직무분류=미상,395.000,"69,747.000",0.566


In [14]:

def competition_check(dataset: str, df_: pd.DataFrame, denominator: str, component_cols: list[str], filter_expr: pd.Series | None = None) -> pd.DataFrame:
    work = df_.copy()
    if filter_expr is not None:
        work = work.loc[filter_expr].copy()
    denom = pd.to_numeric(work[denominator], errors="coerce")
    component_sum = work[component_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)
    residual = denom - component_sum
    out = work[[col for col in ["구분", "계열", "산업분류", "직무분류"] if col in work.columns]].copy()
    out.insert(0, "dataset", dataset)
    out["denominator"] = denom
    out["component_sum"] = component_sum
    out["residual"] = residual
    out["residual_abs"] = residual.abs()
    out["residual_pct"] = residual / denom.replace(0, np.nan) * 100
    out["check_result"] = np.where(out["residual_abs"].le(1e-9), "PASS", "CHECK")
    out["component_cols"] = ", ".join(component_cols)
    return out

company_comp_check = competition_check("df_company", df_company, "분석대상자", company_count_cols)
salary_comp_check = competition_check("df_salary", df_salary, "분석대상자", salary_bucket_cols)
industry_row_check = competition_check("df_industry", df_industry, "분석대상자", [col for col in MAJORS if col in df_industry.columns])
cert_job_degree_check = competition_check("df_cert_job", df_cert_job, "고등교육기관", ["학부", "대학원"])

# 산업 long 기준: 각 계열 안에서 산업분류별 수의 합이 전체 행과 맞는지 본다. 전체 행은 분모라 제외한다.
industry_major_check_rows = []
industry_total_row = df_industry[df_industry["산업분류"].astype("string").eq("전체")]
industry_components = df_industry[~df_industry["산업분류"].astype("string").eq("전체")]
for major in MAJORS:
    denominator = float(industry_total_row[major].iloc[0])
    component_sum = float(industry_components[major].sum())
    industry_major_check_rows.append(
        {
            "dataset": "df_industry_long",
            "계열": major,
            "denominator": denominator,
            "component_sum": component_sum,
            "residual": denominator - component_sum,
            "residual_abs": abs(denominator - component_sum),
            "residual_pct": (denominator - component_sum) / denominator * 100 if denominator else np.nan,
            "check_result": "PASS" if abs(denominator - component_sum) <= 1e-9 else "CHECK",
            "component_cols": "산업분류 전체 제외 모든 행",
        }
    )
industry_major_check = pd.DataFrame(industry_major_check_rows)

# 자격증 계열 자료는 경쟁형 다중 컬럼이 아니라 이항/강도 구조다. 취득률과 1인당 취득수 산식 검산을 별도로 한다.
cert_formula_check = df_cert_major[["구분", "계열", "분석대상자", "자격증취득자", "자격증 취득률", "자격증취득수", "1인당 자격증 취득수"]].copy()
cert_formula_check["미취득자"] = cert_formula_check["분석대상자"] - cert_formula_check["자격증취득자"]
cert_formula_check["자격증취득률_재계산"] = cert_formula_check["자격증취득자"] / cert_formula_check["분석대상자"].replace(0, np.nan) * 100
cert_formula_check["취득률_반올림오차"] = cert_formula_check["자격증 취득률"] - cert_formula_check["자격증취득률_재계산"]
cert_formula_check["1인당취득수_재계산"] = cert_formula_check["자격증취득수"] / cert_formula_check["자격증취득자"].replace(0, np.nan)
cert_formula_check["1인당취득수_반올림오차"] = cert_formula_check["1인당 자격증 취득수"] - cert_formula_check["1인당취득수_재계산"]
cert_formula_check["check_result"] = np.where(
    cert_formula_check[["취득률_반올림오차", "1인당취득수_반올림오차"]].abs().max(axis=1).le(0.1),
    "PASS_rounding",
    "CHECK",
)

all_comp_checks = pd.concat(
    [company_comp_check, salary_comp_check, industry_row_check, cert_job_degree_check, industry_major_check],
    ignore_index=True,
    sort=False,
)
competition_summary = (
    all_comp_checks.groupby("dataset", as_index=False)
    .agg(
        rows=("check_result", "size"),
        pass_rows=("check_result", lambda s: int((s == "PASS").sum())),
        check_rows=("check_result", lambda s: int((s != "PASS").sum())),
        max_abs_residual=("residual_abs", "max"),
    )
)
competition_summary["interpretation"] = np.where(
    competition_summary["check_rows"].eq(0),
    "경쟁형 구성 합계가 분모와 일치. 0은 결측 대체값이 아니라 관측된 0으로 유지",
    "구성 합계와 분모 불일치 행 점검 필요",
)

join_audit = pd.DataFrame(
    [
        {
            "sample": "df_integrated_major",
            "source": "salary_major_features",
            "expected_keys": len(MAJORS),
            "matched_keys": int(df_integrated_major["평균소득만원"].notna().sum()),
            "missing_after_merge_cells": int(df_integrated_major.isna().sum().sum()),
            "decision": "merge 후 NULL 없음" if df_integrated_major.isna().sum().sum() == 0 else "merge 후 NULL 점검 필요",
        },
        {
            "sample": "df_integrated_degree_major",
            "source": "degree_salary+company+cert",
            "expected_keys": len(MAIN_GROUPS) * len(MAJORS),
            "matched_keys": len(df_integrated_degree_major.dropna(subset=["평균소득만원"])),
            "missing_after_merge_cells": int(df_integrated_degree_major.isna().sum().sum()),
            "decision": "merge 후 NULL 없음" if df_integrated_degree_major.isna().sum().sum() == 0 else "merge 후 NULL 점검 필요",
        },
    ]
)

display(Markdown("### 10-3. 경쟁형 구성 합계 검산"))
display(competition_summary)
display(Markdown("### 경쟁형 구성 불일치 행"))
display(all_comp_checks[all_comp_checks["check_result"] != "PASS"] if (all_comp_checks["check_result"] != "PASS").any() else pd.DataFrame([{"status": "모든 경쟁형 구성 합계 PASS"}]))
display(Markdown("### 자격증 계열 자료 산식 검산: 취득률/1인당 취득수"))
display(cert_formula_check.sort_values(["check_result", "구분", "계열"]).head(40))
display(Markdown("### 통합 샘플 merge 후 NULL 점검"))
display(join_audit)


### 10-3. 경쟁형 구성 합계 검산

,dataset,rows,pass_rows,check_rows,max_abs_residual,interpretation
0,df_cert_job,24,24,0,0.000,경쟁형 구성 합계가 분모와 일치. 0은 결측 대체값이 아니라 관측된 0으로 유지
1,df_company,24,24,0,0.000,경쟁형 구성 합계가 분모와 일치. 0은 결측 대체값이 아니라 관측된 0으로 유지
2,df_industry,22,22,0,0.000,경쟁형 구성 합계가 분모와 일치. 0은 결측 대체값이 아니라 관측된 0으로 유지
3,df_industry_long,7,7,0,0.000,경쟁형 구성 합계가 분모와 일치. 0은 결측 대체값이 아니라 관측된 0으로 유지
4,df_salary,40,40,0,0.000,경쟁형 구성 합계가 분모와 일치. 0은 결측 대체값이 아니라 관측된 0으로 유지


### 경쟁형 구성 불일치 행

,status
0,모든 경쟁형 구성 합계 PASS


### 자격증 계열 자료 산식 검산: 취득률/1인당 취득수

,구분,계열,분석대상자,자격증취득자,자격증 취득률,자격증취득수,1인당 자격증 취득수,미취득자,자격증취득률_재계산,취득률_반올림오차,1인당취득수_재계산,1인당취득수_반올림오차,check_result
4,고등교육기관,공학,86625,52824,61.000,124758,2.400,33801,60.980,0.020,2.362,0.038,PASS_rounding
3,고등교육기관,교육,25602,9683,37.800,18580,1.900,15919,37.821,-0.021,1.919,-0.019,PASS_rounding
2,고등교육기관,사회,82516,37707,45.700,70748,1.900,44809,45.697,0.003,1.876,0.024,PASS_rounding
7,고등교육기관,예체능,25117,8341,33.200,13684,1.600,16776,33.209,-0.009,1.641,-0.041,PASS_rounding
6,고등교육기관,의약,47637,13201,27.700,22329,1.700,34436,27.712,-0.012,1.691,0.009,PASS_rounding
1,고등교육기관,인문,23543,10101,42.900,17997,1.800,13442,42.904,-0.004,1.782,0.018,PASS_rounding
5,고등교육기관,자연,28212,16456,58.300,35351,2.100,11756,58.330,-0.030,2.148,-0.048,PASS_rounding
0,고등교육기관,전체,319252,148313,46.500,303447,2.000,170939,46.456,0.044,2.046,-0.046,PASS_rounding
20,대학원,공학,13849,8529,61.600,21001,2.500,5320,61.586,0.014,2.462,0.038,PASS_rounding
19,대학원,교육,10104,5798,57.400,11549,2.000,4306,57.383,0.017,1.992,0.008,PASS_rounding


### 통합 샘플 merge 후 NULL 점검

,sample,source,expected_keys,matched_keys,missing_after_merge_cells,decision
0,df_integrated_major,salary_major_features,7,7,0,merge 후 NULL 없음
1,df_integrated_degree_major,degree_salary+company+cert,21,21,0,merge 후 NULL 없음


In [15]:

# 원천 코드형 결측/미상은 pandas NULL이 아니므로 별도 플래그를 붙인 분석용 샘플을 만든다.
df_integrated_major_cleancheck = df_integrated_major.copy()
df_integrated_major_cleancheck["has_any_null_after_merge"] = df_integrated_major_cleancheck.isna().any(axis=1)
df_integrated_major_cleancheck["zero_sensitive_note"] = "비율 0은 관측된 0으로 유지; NULL 대체값으로 취급 금지"

industry_unknown_by_major = (
    df_source_unknown_volume[df_source_unknown_volume["dataset"].eq("df_industry")]
    .rename(columns={"dimension": "계열", "unknown_count": "산업분류_결측값_취업자수", "unknown_pct": "산업분류_결측값_비율"})
    [["계열", "산업분류_결측값_취업자수", "산업분류_결측값_비율"]]
    if len(df_source_unknown_volume) else pd.DataFrame(columns=["계열", "산업분류_결측값_취업자수", "산업분류_결측값_비율"])
)
df_integrated_major_cleancheck = df_integrated_major_cleancheck.merge(industry_unknown_by_major, on="계열", how="left")
df_integrated_major_cleancheck["산업분류_결측값_취업자수"] = df_integrated_major_cleancheck["산업분류_결측값_취업자수"].fillna(0)
df_integrated_major_cleancheck["산업분류_결측값_비율"] = df_integrated_major_cleancheck["산업분류_결측값_비율"].fillna(0)
df_integrated_major_cleancheck["source_coded_unknown_flag"] = df_integrated_major_cleancheck["산업분류_결측값_취업자수"].gt(0)

cleaning_decisions = pd.DataFrame(
    [
        {
            "issue": "원자료 주석 행",
            "decision": "분석 행 필터링 후 제거",
            "evidence": "구분/산업분류/직무분류의 유효 분석 레벨만 남김",
            "risk_if_wrong": "주석 텍스트가 숫자 변환 실패/가짜 결측으로 들어감",
        },
        {
            "issue": "0값",
            "decision": "경쟁형 구성 컬럼에서는 관측된 0으로 유지",
            "evidence": "구성 합계가 분모와 일치",
            "risk_if_wrong": "0을 NULL로 바꾸면 특정 계열/산업의 실제 부재가 결측으로 왜곡됨",
        },
        {
            "issue": "pandas NULL",
            "decision": "키/타겟/조인 후 NULL은 실질 결측으로 플래그",
            "evidence": "현재 통합 샘플 merge 후 NULL 없음",
            "risk_if_wrong": "조인 실패를 정상 0으로 오인할 수 있음",
        },
        {
            "issue": "산업분류=결측값",
            "decision": "NULL은 아니지만 원천 코드형 분류 미상으로 별도 비율 산출",
            "evidence": "df_source_unknown_volume에 계열별 결측값 취업자수/비율 산출",
            "risk_if_wrong": "산업 구조 집중도를 과신할 수 있음",
        },
        {
            "issue": "직무분류=미상",
            "decision": "직무별 자격증 EDA에서 원천 코드형 미상으로 별도 관리",
            "evidence": "df_source_unknown_volume에 고등교육기관/학부/대학원별 미상 비율 산출",
            "risk_if_wrong": "직무 자격증 Top 순위 해석에서 분류 미상 규모 누락",
        },
        {
            "issue": "타겟 누수",
            "decision": "평균소득만원을 타겟으로 둘 때 급여구간 비율은 타겟 진단으로 분리",
            "evidence": "target_candidates에서 distribution_target_or_segment_y로 별도 표기",
            "risk_if_wrong": "같은 급여분포에서 나온 컬럼으로 급여를 설명하는 자기참조 발생",
        },
    ]
)

sample_quality_score = pd.DataFrame(
    [
        {
            "sample": "df_integrated_major",
            "unit": "고등교육기관 x 계열",
            "rows": len(df_integrated_major),
            "pandas_null_cells": int(df_integrated_major.isna().sum().sum()),
            "source_coded_unknown_scope": "산업분류 결측값 비율을 별도 피처/주의 플래그로 보유",
            "zero_handling": "0은 관측값 유지",
            "usable_for": "기사형 EDA/타겟 설계/가설 생성",
            "not_usable_for": "n=7 통계 모델 일반화",
        },
        {
            "sample": "df_integrated_degree_major",
            "unit": "구분 x 계열",
            "rows": len(df_integrated_degree_major),
            "pandas_null_cells": int(df_integrated_degree_major.isna().sum().sum()),
            "source_coded_unknown_scope": "산업유형이 빠져 원천 산업 결측은 미포함",
            "zero_handling": "0은 관측값 유지",
            "usable_for": "학부/대학원 구분별 급여-기업유형-자격증 비교",
            "not_usable_for": "산업 믹스 설명",
        },
        {
            "sample": "df_cert_job",
            "unit": "직무분류",
            "rows": len(df_cert_job),
            "pandas_null_cells": int(df_cert_job.isna().sum().sum()),
            "source_coded_unknown_scope": "직무분류=미상 별도 산출",
            "zero_handling": "0은 해당 직무 자격증 취득수 없음으로 유지",
            "usable_for": "직무별 자격증 집중도와 미상 규모 점검",
            "not_usable_for": "계열 통합 직접 조인",
        },
    ]
)

display(Markdown("### 10-4. 클리닝 결정표"))
display(cleaning_decisions)
display(Markdown("### 통합 샘플에 원천 코드형 결측 플래그 반영"))
display(df_integrated_major_cleancheck)
display(Markdown("### 샘플별 사용 가능성/제한"))
display(sample_quality_score)


### 10-4. 클리닝 결정표

,issue,decision,evidence,risk_if_wrong
0,원자료 주석 행,분석 행 필터링 후 제거,구분/산업분류/직무분류의 유효 분석 레벨만 남김,주석 텍스트가 숫자 변환 실패/가짜 결측으로 들어감
1,0값,경쟁형 구성 컬럼에서는 관측된 0으로 유지,구성 합계가 분모와 일치,0을 NULL로 바꾸면 특정 계열/산업의 실제 부재가 결측으로 왜곡됨
2,pandas NULL,키/타겟/조인 후 NULL은 실질 결측으로 플래그,현재 통합 샘플 merge 후 NULL 없음,조인 실패를 정상 0으로 오인할 수 있음
3,산업분류=결측값,NULL은 아니지만 원천 코드형 분류 미상으로 별도 비율 산출,df_source_unknown_volume에 계열별 결측값 취업자수/비율 산출,산업 구조 집중도를 과신할 수 있음
4,직무분류=미상,직무별 자격증 EDA에서 원천 코드형 미상으로 별도 관리,df_source_unknown_volume에 고등교육기관/학부/대학원별 미상 비율 산출,직무 자격증 Top 순위 해석에서 분류 미상 규모 누락
5,타겟 누수,평균소득만원을 타겟으로 둘 때 급여구간 비율은 타겟 진단으로 분리,target_candidates에서 distribution_target_or_seg...,같은 급여분포에서 나온 컬럼으로 급여를 설명하는 자기참조 발생


### 통합 샘플에 원천 코드형 결측 플래그 반영

,계열,분석대상자,평균소득만원,중위소득만원,평균_중위_차이,평균_중위_비율,100만원미만_비율,100~200만원미만_비율,200~300만원미만_비율,300~400만원미만_비율,400만원이상_비율,300만원이상_비율,200만원미만_비율,대기업_비율,중견기업_비율,중소기업_비율,국가및지방_비율,공공기관_비율,비영리_비율,기타_비율,대중견기업비율,공공비영리비율,자격증 취득률,자격증취득자,자격증취득수,1인당 자격증 취득수,상위산업,상위산업비율,상위3산업비율,산업허핀달지수,제조업비율,정보통신업비율,전문과학기술비율,공공행정비율,교육서비스비율,보건사회복지비율,예술스포츠여가비율,log10_평균소득만원,log10_중위소득만원,has_any_null_after_merge,zero_sensitive_note,산업분류_결측값_취업자수,산업분류_결측값_비율,source_coded_unknown_flag
0,인문,23543,316.300,255.900,60.400,1.236,2.880,12.560,48.482,16.005,20.074,36.079,15.440,11.065,5.513,41.176,18.511,2.532,17.585,3.619,16.578,38.627,42.900,10101,17997,1.800,P.교육서비스업,13.350,36.669,0.084,10.661,8.317,7.947,12.658,13.350,7.595,2.094,2.500,2.408,False,비율 0은 관측된 0으로 유지; NULL 대체값으로 취급 금지,656.000,2.786,True
1,사회,82516,364.300,270.700,93.600,1.346,2.197,9.958,45.890,16.044,25.910,41.954,12.155,12.164,6.290,40.393,14.910,3.839,16.311,6.093,18.453,35.060,45.700,37707,70748,1.900,Q.보건·사회복지업,13.178,38.485,0.086,12.264,5.895,9.950,13.042,5.346,13.178,1.138,2.561,2.432,False,비율 0은 관측된 0으로 유지; NULL 대체값으로 취급 금지,"1,723.000",2.088,True
2,교육,25602,329.700,290.600,39.100,1.135,1.578,9.226,42.497,21.494,25.205,46.699,10.804,1.965,0.953,13.159,41.618,0.980,18.522,22.803,2.918,61.120,37.800,9683,18580,1.900,P.교육서비스업,43.528,74.701,0.246,2.273,1.668,2.062,10.444,43.528,20.729,0.777,2.518,2.463,False,비율 0은 관측된 0으로 유지; NULL 대체값으로 취급 금지,"2,605.000",10.175,True
3,공학,86625,361.700,308.300,53.400,1.173,1.067,6.019,39.984,22.361,30.570,52.930,7.086,23.100,10.385,49.729,7.237,3.508,5.315,0.726,33.485,16.060,61.000,52824,124758,2.400,C.제조업,38.123,63.734,0.194,38.123,12.317,13.294,6.185,2.192,0.831,0.428,2.558,2.489,False,비율 0은 관측된 0으로 유지; NULL 대체값으로 취급 금지,813.000,0.939,True
4,자연,28212,311.000,260.000,51.000,1.196,2.166,10.758,51.673,17.875,17.528,35.403,12.924,14.536,7.071,50.670,9.819,3.506,12.778,1.620,21.608,26.102,58.300,16456,35351,2.100,C.제조업,24.965,53.339,0.128,24.965,4.062,17.950,7.231,4.927,4.597,1.439,2.493,2.415,False,비율 0은 관측된 0으로 유지; NULL 대체값으로 취급 금지,458.000,1.623,True
5,의약,47637,360.200,294.400,65.800,1.224,0.940,5.395,45.305,28.925,19.434,48.359,6.335,1.677,1.064,46.397,4.121,4.263,41.959,0.519,2.742,50.343,27.700,13201,22329,1.700,Q.보건·사회복지업,77.280,85.209,0.603,4.096,0.739,2.966,3.205,2.095,77.280,0.447,2.557,2.469,False,비율 0은 관측된 0으로 유지; NULL 대체값으로 취급 금지,373.000,0.783,True
6,예체능,25117,245.100,220.000,25.100,1.114,4.531,22.148,56.906,9.328,7.087,16.415,26.679,7.959,4.423,69.328,6.517,1.063,9.746,0.963,12.382,17.327,33.200,8341,13684,1.600,G.도·소매업,13.660,36.461,0.085,12.947,9.854,8.576,4.790,9.424,4.770,4.993,2.389,2.342,False,비율 0은 관측된 0으로 유지; NULL 대체값으로 취급 금지,667.000,2.656,True


### 샘플별 사용 가능성/제한

,sample,unit,rows,pandas_null_cells,source_coded_unknown_scope,zero_handling,usable_for,not_usable_for
0,df_integrated_major,고등교육기관 x 계열,7,0,산업분류 결측값 비율을 별도 피처/주의 플래그로 보유,0은 관측값 유지,기사형 EDA/타겟 설계/가설 생성,n=7 통계 모델 일반화
1,df_integrated_degree_major,구분 x 계열,21,0,산업유형이 빠져 원천 산업 결측은 미포함,0은 관측값 유지,학부/대학원 구분별 급여-기업유형-자격증 비교,산업 믹스 설명
2,df_cert_job,직무분류,24,0,직무분류=미상 별도 산출,0은 해당 직무 자격증 취득수 없음으로 유지,직무별 자격증 집중도와 미상 규모 점검,계열 통합 직접 조인


In [16]:

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

plot_missing = real_missing_summary.sort_values("actual_missing_cells", ascending=False).copy()
axes[0].barh(plot_missing["dataset"], plot_missing["actual_missing_cells"], color="#E45756", label="실질 결측/원천 미상 셀")
axes[0].barh(plot_missing["dataset"], plot_missing["pandas_null_cells"], color="#4C78A8", alpha=0.7, label="pandas NULL 셀")
axes[0].set_title("데이터셋별 NULL vs 실질 결측")
axes[0].set_xlabel("cell count")
axes[0].legend(loc="lower right")

if len(df_source_unknown_volume):
    unknown_plot = df_source_unknown_volume.copy()
    unknown_plot["label"] = unknown_plot["dataset"] + " / " + unknown_plot["dimension"].astype(str)
    unknown_plot = unknown_plot.sort_values("unknown_pct", ascending=True)
    axes[1].barh(unknown_plot["label"], unknown_plot["unknown_pct"], color="#F58518")
    axes[1].set_title("원천 코드형 미상/결측값 비율")
    axes[1].set_xlabel("비율(%)")
else:
    axes[1].text(0.5, 0.5, "원천 코드형 미상/결측값 없음", ha="center", va="center")
    axes[1].set_axis_off()

display(fig)
plt.close(fig)


<Figure size 1500x500 with 2 Axes>

## 11. 확정 결정: `df_integrated_major` 채택과 직무-산업 후보 매핑 분리

이 섹션은 분석 방향을 확정한다.

- 메인 통합 샘플은 `df_integrated_major` (`고등교육기관 x 계열`, 7행)로 고정한다.
- `df_cert_job`은 아직 조인하지 않는다.
- 대신 `직무분류 -> 산업분류` 후보 매핑표를 별도 산출물로 만들고, 기존 `P2_학과별_A비율_대학아님.csv`의 `학과_정규화`/`raw_names` alias 가이드를 근거로 남긴다.
- 이 매핑표는 후속 검토용이며, 현재 평균소득 EDA의 피처로 사용하지 않는다.


In [17]:

G0_TABLE_DIR = BASE_DIR / "eda/tables"
G0_TABLE_DIR.mkdir(parents=True, exist_ok=True)
DEPT_GUIDE_PATH = BASE_DIR.parent / "P2_학과별_A비율_대학아님.csv"
JOB_TO_INDUSTRY_MAPPING_PATH = G0_TABLE_DIR / "23_cert_job_to_industry_mapping_candidates.csv"
CLEANING_DECISION_REPORT_PATH = BASE_DIR / "eda/P2_G0_major_cleaning_decision_report.md"

if not DEPT_GUIDE_PATH.exists():
    raise FileNotFoundError(DEPT_GUIDE_PATH)

dept_guide = pd.read_csv(DEPT_GUIDE_PATH, encoding="utf-8-sig", low_memory=False)
required_guide_cols = ["학과_정규화", "raw_names", "students", "schools", "a_band_pct_weighted"]
missing_guide_cols = [col for col in required_guide_cols if col not in dept_guide.columns]
if missing_guide_cols:
    raise KeyError(f"학과명 매칭 가이드 필수 컬럼 누락: {missing_guide_cols}")

dept_guide["학과_정규화"] = dept_guide["학과_정규화"].astype("string").str.strip()
dept_guide["raw_names"] = dept_guide["raw_names"].astype("string").fillna("").str.strip()
dept_guide["students"] = pd.to_numeric(dept_guide["students"], errors="coerce").fillna(0)
dept_guide["schools"] = pd.to_numeric(dept_guide["schools"], errors="coerce").fillna(0)
dept_guide["a_band_pct_weighted"] = pd.to_numeric(dept_guide["a_band_pct_weighted"], errors="coerce")

JOB_MAPPING_SPECS = {
    "전체": {
        "primary_industry": "",
        "candidate_industries": "",
        "guide_terms": [],
        "mapping_type": "aggregate_exclude",
        "confidence": "exclude",
        "review_needed": True,
        "decision_note": "집계 행이므로 산업분류 매핑 대상에서 제외",
    },
    "경영·회계·사무": {
        "primary_industry": "M.전문·과학·기술업",
        "candidate_industries": "M.전문·과학·기술업 | N.사업시설·지원업 | K.금융·보험업 | O.공공행정 | G.도·소매업",
        "guide_terms": ["경영", "회계", "세무", "사무", "비서"],
        "mapping_type": "cross_industry_support_function",
        "confidence": "low",
        "review_needed": True,
        "decision_note": "직무가 산업이 아니라 기능에 가까워 단일 산업으로 확정하면 위험",
    },
    "교육·자연과학·사회과학": {
        "primary_industry": "P.교육서비스업",
        "candidate_industries": "P.교육서비스업 | M.전문·과학·기술업",
        "guide_terms": ["교육", "자연과학", "사회과학", "수학", "물리", "화학", "생명과학", "사회학"],
        "mapping_type": "mixed_education_research",
        "confidence": "low",
        "review_needed": True,
        "decision_note": "건수가 매우 작고 교육/연구 기능이 섞여 후속 검토 필요",
    },
    "보건·의료": {
        "primary_industry": "Q.보건·사회복지업",
        "candidate_industries": "Q.보건·사회복지업",
        "guide_terms": ["보건", "의료", "간호", "임상병리", "물리치료", "치위생", "방사선", "응급구조"],
        "mapping_type": "direct_domain_match",
        "confidence": "high",
        "review_needed": False,
        "decision_note": "학과명 가이드의 보건/의료/간호 계열과 산업 대분류가 직접 대응",
    },
    "사회복지·종교": {
        "primary_industry": "Q.보건·사회복지업",
        "candidate_industries": "Q.보건·사회복지업 | S.협회·기타서비스업",
        "guide_terms": ["사회복지", "복지", "종교", "신학"],
        "mapping_type": "mixed_social_religious_service",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "사회복지는 Q가 강하지만 종교는 S 후보가 있어 분리 검토 필요",
    },
    "문화·예술·디자인·방송": {
        "primary_industry": "R.예술·스포츠·여가업",
        "candidate_industries": "R.예술·스포츠·여가업 | J.정보통신업 | M.전문·과학·기술업",
        "guide_terms": ["문화", "예술", "디자인", "방송", "미디어", "영상", "콘텐츠", "공연", "음악", "미술"],
        "mapping_type": "creative_content_mixed",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "콘텐츠/방송은 J, 디자인은 M/R로 갈 수 있어 후보형 유지",
    },
    "운전·운송": {
        "primary_industry": "H.운수업",
        "candidate_industries": "H.운수업",
        "guide_terms": ["운송", "항공운송", "해상운송", "물류", "교통"],
        "mapping_type": "direct_domain_match",
        "confidence": "high",
        "review_needed": False,
        "decision_note": "직무명과 산업 대분류가 직접 대응",
    },
    "영업·판매": {
        "primary_industry": "G.도·소매업",
        "candidate_industries": "G.도·소매업",
        "guide_terms": ["영업", "판매", "유통", "무역", "마케팅"],
        "mapping_type": "direct_domain_match",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "판매 기능은 도소매가 주 후보지만 업종 횡단 기능이므로 검토 필요",
    },
    "이용·숙박·여행·오락·스포츠": {
        "primary_industry": "I.숙박·음식점업",
        "candidate_industries": "I.숙박·음식점업 | R.예술·스포츠·여가업 | S.협회·기타서비스업",
        "guide_terms": ["숙박", "여행", "관광", "오락", "스포츠", "체육", "레저", "뷰티", "미용"],
        "mapping_type": "personal_tourism_sports_service_mixed",
        "confidence": "low",
        "review_needed": True,
        "decision_note": "숙박/여행/스포츠/이용 서비스가 섞여 단일 산업 확정 금지",
    },
    "음식서비스": {
        "primary_industry": "I.숙박·음식점업",
        "candidate_industries": "I.숙박·음식점업",
        "guide_terms": ["음식", "외식", "조리", "호텔조리", "식음료", "푸드"],
        "mapping_type": "direct_domain_match",
        "confidence": "high",
        "review_needed": False,
        "decision_note": "음식서비스는 숙박·음식점업과 직접 대응",
    },
    "건설": {
        "primary_industry": "F.건설업",
        "candidate_industries": "F.건설업",
        "guide_terms": ["건설", "건축", "토목", "도시", "건설환경"],
        "mapping_type": "direct_domain_match",
        "confidence": "high",
        "review_needed": False,
        "decision_note": "건설/건축/토목 학과명 근거가 명확",
    },
    "광업자원": {
        "primary_industry": "B.광업",
        "candidate_industries": "B.광업 | A.농림어업 | M.전문·과학·기술업",
        "guide_terms": ["광업", "자원", "에너지자원", "지질", "자원공"],
        "mapping_type": "small_count_resource_domain",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "직무 건수가 작고 자원/에너지/연구 성격이 섞여 후보형 유지",
    },
    "기계": {
        "primary_industry": "C.제조업",
        "candidate_industries": "C.제조업",
        "guide_terms": ["기계", "자동차", "메카트로닉스", "로봇"],
        "mapping_type": "manufacturing_domain_match",
        "confidence": "high",
        "review_needed": False,
        "decision_note": "기계 계열 자격은 제조업 대분류 후보가 가장 강함",
    },
    "재료": {
        "primary_industry": "C.제조업",
        "candidate_industries": "C.제조업",
        "guide_terms": ["재료", "신소재", "금속", "세라믹", "나노"],
        "mapping_type": "manufacturing_domain_match",
        "confidence": "high",
        "review_needed": False,
        "decision_note": "재료/신소재는 제조업 소재 분야로 대응",
    },
    "화학": {
        "primary_industry": "C.제조업",
        "candidate_industries": "C.제조업 | E.수도·하수·폐기물업 | M.전문·과학·기술업",
        "guide_terms": ["화학", "화공", "생명화학", "화학공"],
        "mapping_type": "manufacturing_with_research_environment_candidates",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "제조업이 주 후보지만 연구/환경 자격과 일부 겹칠 수 있음",
    },
    "섬유·의복": {
        "primary_industry": "C.제조업",
        "candidate_industries": "C.제조업 | R.예술·스포츠·여가업",
        "guide_terms": ["섬유", "의복", "패션", "의류"],
        "mapping_type": "manufacturing_design_mixed",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "섬유 생산은 제조업, 패션디자인은 예술/전문서비스와 겹침",
    },
    "전기·전자": {
        "primary_industry": "C.제조업",
        "candidate_industries": "C.제조업 | D.전기·가스·증기업",
        "guide_terms": ["전기", "전자", "전기전자", "반도체"],
        "mapping_type": "manufacturing_energy_mixed",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "전자 제조가 주 후보이나 전기 설비/에너지 산업과 일부 겹침",
    },
    "정보통신": {
        "primary_industry": "J.정보통신업",
        "candidate_industries": "J.정보통신업 | C.제조업",
        "guide_terms": ["정보통신", "컴퓨터", "소프트웨어", "AI", "데이터", "보안", "인공지능"],
        "mapping_type": "direct_ict_match",
        "confidence": "high",
        "review_needed": False,
        "decision_note": "학과명 가이드에서 컴퓨터/소프트웨어/정보통신 근거가 강함",
    },
    "식품가공": {
        "primary_industry": "C.제조업",
        "candidate_industries": "C.제조업 | I.숙박·음식점업",
        "guide_terms": ["식품", "식품공", "식품가공", "식품영양", "푸드"],
        "mapping_type": "food_manufacturing_main",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "식품가공은 제조업이 주 후보이나 서비스/영양 학과와 일부 겹침",
    },
    "인쇄·목재·가구·공예": {
        "primary_industry": "C.제조업",
        "candidate_industries": "C.제조업 | R.예술·스포츠·여가업",
        "guide_terms": ["인쇄", "목재", "가구", "공예", "목조", "출판"],
        "mapping_type": "manufacturing_craft_mixed",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "제조업 기반이지만 공예/디자인 계열과 겹쳐 후보형 유지",
    },
    "농림어업": {
        "primary_industry": "A.농림어업",
        "candidate_industries": "A.농림어업",
        "guide_terms": ["농", "농림", "어업", "수산", "동물자원", "원예", "산림"],
        "mapping_type": "direct_domain_match",
        "confidence": "high",
        "review_needed": False,
        "decision_note": "농림어업 산업 대분류와 직접 대응",
    },
    "안전관리": {
        "primary_industry": "",
        "candidate_industries": "O.공공행정 | N.사업시설·지원업 | C.제조업 | F.건설업",
        "guide_terms": ["안전", "산업안전", "소방", "방재", "보건환경안전", "보안"],
        "mapping_type": "cross_industry_risk_management",
        "confidence": "low",
        "review_needed": True,
        "decision_note": "안전관리는 여러 산업에 걸친 기능이라 현재 단계에서 primary 확정 금지",
    },
    "환경·에너지": {
        "primary_industry": "E.수도·하수·폐기물업",
        "candidate_industries": "E.수도·하수·폐기물업 | D.전기·가스·증기업 | M.전문·과학·기술업",
        "guide_terms": ["환경", "에너지", "신재생", "폐기물", "수질", "대기", "에너지공"],
        "mapping_type": "environment_energy_mixed",
        "confidence": "medium",
        "review_needed": True,
        "decision_note": "환경과 에너지가 결합되어 E/D/M 후보를 모두 유지",
    },
    "미상": {
        "primary_industry": "결측값",
        "candidate_industries": "결측값",
        "guide_terms": [],
        "mapping_type": "source_coded_unknown",
        "confidence": "unknown",
        "review_needed": True,
        "decision_note": "원천 코드형 미상. 산업분류로 배정하지 말고 결측성 플래그로 유지",
    },
}


def guide_evidence(terms: list[str]) -> dict[str, object]:
    if not terms:
        return {
            "guide_match_dept_n": 0,
            "guide_match_school_sum": 0,
            "guide_match_student_sum": 0,
            "guide_match_top_departments": "",
            "guide_terms": "",
        }
    pattern = "|".join(re.escape(term) for term in terms if term)
    mask = dept_guide["학과_정규화"].str.contains(pattern, regex=True, na=False) | dept_guide["raw_names"].str.contains(pattern, regex=True, na=False)
    matched = dept_guide[mask].copy().sort_values("students", ascending=False)
    top = matched.head(6)
    top_text = " | ".join([f"{row['학과_정규화']}({int(row['students']):,})" for _, row in top.iterrows()])
    return {
        "guide_match_dept_n": int(mask.sum()),
        "guide_match_school_sum": int(matched["schools"].sum()) if len(matched) else 0,
        "guide_match_student_sum": int(matched["students"].sum()) if len(matched) else 0,
        "guide_match_top_departments": top_text,
        "guide_terms": ", ".join(terms),
    }

mapping_rows = []
job_total = float(df_cert_job.loc[df_cert_job["직무분류"].astype("string").eq("전체"), "고등교육기관"].iloc[0])
for _, row in df_cert_job.iterrows():
    job = str(row["직무분류"]).strip()
    spec = JOB_MAPPING_SPECS.get(
        job,
        {
            "primary_industry": "",
            "candidate_industries": "",
            "guide_terms": [job],
            "mapping_type": "unspecified_review_required",
            "confidence": "unknown",
            "review_needed": True,
            "decision_note": "사전 정의 없는 직무분류. 수동 검토 필요",
        },
    )
    evidence = guide_evidence(spec["guide_terms"])
    mapping_rows.append(
        {
            "직무분류": job,
            "고등교육기관_자격증취득수": row["고등교육기관"],
            "학부_자격증취득수": row["학부"],
            "대학원_자격증취득수": row["대학원"],
            "직무비율_pct": row["고등교육기관"] / job_total * 100 if job_total else np.nan,
            "primary_industry_candidate": spec["primary_industry"],
            "industry_candidates": spec["candidate_industries"],
            "mapping_type": spec["mapping_type"],
            "confidence": spec["confidence"],
            "review_needed": spec["review_needed"],
            "join_now": False,
            "current_analysis_policy": "별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음",
            **evidence,
            "decision_note": spec["decision_note"],
        }
    )

df_job_to_industry_mapping = pd.DataFrame(mapping_rows)
df_job_to_industry_mapping.to_csv(JOB_TO_INDUSTRY_MAPPING_PATH, index=False, encoding="utf-8-sig")

mapping_summary = pd.DataFrame(
    [
        {"metric": "mapping_rows", "value": len(df_job_to_industry_mapping), "note": "df_cert_job의 전체/미상 포함 행 수"},
        {"metric": "valid_job_rows_excluding_total", "value": int((~df_job_to_industry_mapping["직무분류"].eq("전체")).sum()), "note": "전체 제외 직무분류 수"},
        {"metric": "join_now_true_rows", "value": int(df_job_to_industry_mapping["join_now"].sum()), "note": "현재 조인 금지 확인"},
        {"metric": "review_needed_rows", "value": int(df_job_to_industry_mapping["review_needed"].sum()), "note": "복합/기능형 직무라 수동 검토 필요한 행"},
        {"metric": "output_path", "value": str(JOB_TO_INDUSTRY_MAPPING_PATH), "note": "별도 후보 매핑표 CSV"},
    ]
)

display(Markdown("### 11-1. 직무분류 -> 산업분류 후보 매핑표 생성"))
display(mapping_summary)
display(df_job_to_industry_mapping)


### 11-1. 직무분류 -> 산업분류 후보 매핑표 생성

,metric,value,note
0,mapping_rows,24,df_cert_job의 전체/미상 포함 행 수
1,valid_job_rows_excluding_total,23,전체 제외 직무분류 수
2,join_now_true_rows,0,현재 조인 금지 확인
3,review_needed_rows,16,복합/기능형 직무라 수동 검토 필요한 행
4,output_path,/home/sieg/projects-wsl/SBS_dataScience/workbo...,별도 후보 매핑표 CSV


,직무분류,고등교육기관_자격증취득수,학부_자격증취득수,대학원_자격증취득수,직무비율_pct,primary_industry_candidate,industry_candidates,mapping_type,confidence,review_needed,join_now,current_analysis_policy,guide_match_dept_n,guide_match_school_sum,guide_match_student_sum,guide_match_top_departments,guide_terms,decision_note
0,전체,"303,447.000","233,700.000","69,747.000",100.000,,,aggregate_exclude,exclude,True,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,0,0,0,,,집계 행이므로 산업분류 매핑 대상에서 제외
1,경영·회계·사무,"127,952.000","92,349.000","35,603.000",42.166,M.전문·과학·기술업,M.전문·과학·기술업 | N.사업시설·지원업 | K.금융·보험업 | O.공공행정 |...,cross_industry_support_function,low,True,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,298,757,893316,"경영(370,552) | 경영학(59,542) | 글로벌경영(33,531) | 관광...","경영, 회계, 세무, 사무, 비서",직무가 산업이 아니라 기능에 가까워 단일 산업으로 확정하면 위험
2,교육·자연과학·사회과학,11.000,4.000,7.000,0.004,P.교육서비스업,P.교육서비스업 | M.전문·과학·기술업,mixed_education_research,low,True,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,463,1539,1236256,"유아교육과(96,382) | 물리치료(79,375) | 화학공(55,905) | 교...","교육, 자연과학, 사회과학, 수학, 물리, 화학, 생명과학, 사회학",건수가 매우 작고 교육/연구 기능이 섞여 후속 검토 필요
3,보건·의료,938.000,221.000,717.000,0.309,Q.보건·사회복지업,Q.보건·사회복지업,direct_domain_match,high,False,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,101,437,990409,"간호(611,529) | 물리치료(79,375) | 임상병리(49,311) | 치위...","보건, 의료, 간호, 임상병리, 물리치료, 치위생, 방사선, 응급구조",학과명 가이드의 보건/의료/간호 계열과 산업 대분류가 직접 대응
4,사회복지·종교,"1,247.000",715.000,532.000,0.411,Q.보건·사회복지업,Q.보건·사회복지업 | S.협회·기타서비스업,mixed_social_religious_service,medium,True,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,128,384,479724,"사회복지(283,988) | 신(30,231) | 청소년교육복지상담(16,541) ...","사회복지, 복지, 종교, 신학",사회복지는 Q가 강하지만 종교는 S 후보가 있어 분리 검토 필요
5,문화·예술·디자인·방송,"3,931.000","3,209.000",722.000,1.295,R.예술·스포츠·여가업,R.예술·스포츠·여가업 | J.정보통신업 | M.전문·과학·기술업,creative_content_mixed,medium,True,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,730,1360,1124787,"음악(48,216) | 미디어커뮤니케이션(41,174) | 디자인(35,556) |...","문화, 예술, 디자인, 방송, 미디어, 영상, 콘텐츠, 공연, 음악, 미술","콘텐츠/방송은 J, 디자인은 M/R로 갈 수 있어 후보형 유지"
6,운전·운송,238.000,213.000,25.000,0.078,H.운수업,H.운수업,direct_domain_match,high,False,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,61,89,56351,"항공교통물류(6,233) | 국제물류(5,029) | 해상운송(3,935) | 무역...","운송, 항공운송, 해상운송, 물류, 교통",직무명과 산업 대분류가 직접 대응
7,영업·판매,646.000,426.000,220.000,0.213,G.도·소매업,G.도·소매업,direct_domain_match,medium,True,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,64,112,95716,"무역(27,021) | 국제무역(8,749) | 무역학(4,999) | 유통마케팅(...","영업, 판매, 유통, 무역, 마케팅",판매 기능은 도소매가 주 후보지만 업종 횡단 기능이므로 검토 필요
8,이용·숙박·여행·오락·스포츠,"6,234.000","5,663.000",571.000,2.054,I.숙박·음식점업,I.숙박·음식점업 | R.예술·스포츠·여가업 | S.협회·기타서비스업,personal_tourism_sports_service_mixed,low,True,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,263,614,507225,"체육(47,095) | 스포츠과(37,580) | 체육교육과(35,828) | 관광...","숙박, 여행, 관광, 오락, 스포츠, 체육, 레저, 뷰티, 미용",숙박/여행/스포츠/이용 서비스가 섞여 단일 산업 확정 금지
9,음식서비스,"13,790.000","11,524.000","2,266.000",4.544,I.숙박·음식점업,I.숙박·음식점업,direct_domain_match,high,False,False,별도 후보 매핑표로만 보관. df_integrated_major에는 조인하지 않음,83,127,78489,"호텔관광경영(6,804) | 외식조리(4,802) | 호텔조리(3,606) | 호텔...","음식, 외식, 조리, 호텔조리, 식음료, 푸드",음식서비스는 숙박·음식점업과 직접 대응


In [18]:

final_sample_decision = pd.DataFrame(
    [
        {
            "decision_item": "main_analysis_table",
            "decision": "df_integrated_major",
            "status": "CONFIRMED",
            "reason": "기업유형 + 계열별 자격증 + 초임급여 + 산업유형을 모두 포함할 수 있는 유일한 안전 단위",
        },
        {
            "decision_item": "analysis_unit",
            "decision": "고등교육기관 x 계열, 7행",
            "status": "CONFIRMED",
            "reason": "산업유형 파일에 구분 컬럼이 없으므로 학부/대학원 확장 샘플에는 산업 피처를 붙이면 안 됨",
        },
        {
            "decision_item": "target",
            "decision": "평균소득만원 primary, 중위소득만원 secondary",
            "status": "CONFIRMED",
            "reason": "초임급여 수준을 직접 설명하는 연속형 타겟. 급여구간 비율은 타겟 진단으로 분리",
        },
        {
            "decision_item": "cert_job_join",
            "decision": "조인하지 않음",
            "status": "CONFIRMED",
            "reason": "직무분류는 계열/산업분류와 직접 키가 다르므로 현재 통합 샘플에 넣지 않음",
        },
        {
            "decision_item": "job_to_industry_mapping",
            "decision": str(JOB_TO_INDUSTRY_MAPPING_PATH),
            "status": "CANDIDATE_TABLE_CREATED",
            "reason": "후속 검토용 후보 매핑표만 별도 생성. 현재 모델/EDA 피처로 미사용",
        },
        {
            "decision_item": "cleaning_status",
            "decision": "current_sample_cleaning_complete_with_flags",
            "status": "CONDITIONAL_COMPLETE",
            "reason": "주석 행 제거, 숫자 변환, 경쟁형 구성 검산, merge NULL 점검 완료. 원천 코드형 결측값/미상은 삭제하지 않고 플래그로 유지",
        },
    ]
)

report_text = f"""# P2_G0 클리닝/샘플 결정 리포트

## 확정 결정

- 메인 분석 테이블: `df_integrated_major`
- 분석 단위: `고등교육기관 x 계열`, 7행
- 1차 타겟: `평균소득만원`
- 보조 타겟: `중위소득만원`
- 급여구간 비율: 타겟 설명 피처가 아니라 타겟 진단/별도 타겟 후보
- 직무별 자격증: 현재 통합 샘플에 조인하지 않음

## 클리닝 상태 판단

현재 `df_integrated_major` 기준 데이터 클리닝은 조건부 완료로 본다.

완료된 점검:

- 원자료 주석/설명 행 제거
- 텍스트 키 정리와 숫자형 변환
- 기업유형, 초임급여 구간, 산업유형, 직무별 자격증 경쟁형 구성 합계 검산
- 통합 샘플 merge 후 pandas NULL 없음 확인
- 원천 코드형 `산업분류=결측값`, `직무분류=미상`을 pandas NULL과 분리

남겨야 할 제한:

- 산업분류 `결측값`은 삭제하지 않고 미상 비율로 주석 처리한다.
- 특히 교육 계열은 산업분류 미상 비율이 높아 산업 믹스 해석에 제한이 있다.
- 직무별 자격증은 계열/산업과 직접 조인하지 않는다.
- `직무분류 -> 산업분류` 매핑표는 후보표이며 수동 검토 전에는 분석 피처로 사용하지 않는다.

## 산출물

- 노트북: `{NOTEBOOK_PATH}`
- 직무-산업 후보 매핑표: `{JOB_TO_INDUSTRY_MAPPING_PATH}`
- 리포트: `{CLEANING_DECISION_REPORT_PATH}`
"""
CLEANING_DECISION_REPORT_PATH.write_text(report_text, encoding="utf-8")

display(Markdown("### 11-2. 최종 분석/클리닝 결정표"))
display(final_sample_decision)
display(Markdown(report_text))


### 11-2. 최종 분석/클리닝 결정표

,decision_item,decision,status,reason
0,main_analysis_table,df_integrated_major,CONFIRMED,기업유형 + 계열별 자격증 + 초임급여 + 산업유형을 모두 포함할 수 있는 유일한 ...
1,analysis_unit,"고등교육기관 x 계열, 7행",CONFIRMED,산업유형 파일에 구분 컬럼이 없으므로 학부/대학원 확장 샘플에는 산업 피처를 붙이면...
2,target,"평균소득만원 primary, 중위소득만원 secondary",CONFIRMED,초임급여 수준을 직접 설명하는 연속형 타겟. 급여구간 비율은 타겟 진단으로 분리
3,cert_job_join,조인하지 않음,CONFIRMED,직무분류는 계열/산업분류와 직접 키가 다르므로 현재 통합 샘플에 넣지 않음
4,job_to_industry_mapping,/home/sieg/projects-wsl/SBS_dataScience/workbo...,CANDIDATE_TABLE_CREATED,후속 검토용 후보 매핑표만 별도 생성. 현재 모델/EDA 피처로 미사용
5,cleaning_status,current_sample_cleaning_complete_with_flags,CONDITIONAL_COMPLETE,"주석 행 제거, 숫자 변환, 경쟁형 구성 검산, merge NULL 점검 완료. 원..."


# P2_G0 클리닝/샘플 결정 리포트

## 확정 결정

- 메인 분석 테이블: `df_integrated_major`
- 분석 단위: `고등교육기관 x 계열`, 7행
- 1차 타겟: `평균소득만원`
- 보조 타겟: `중위소득만원`
- 급여구간 비율: 타겟 설명 피처가 아니라 타겟 진단/별도 타겟 후보
- 직무별 자격증: 현재 통합 샘플에 조인하지 않음

## 클리닝 상태 판단

현재 `df_integrated_major` 기준 데이터 클리닝은 조건부 완료로 본다.

완료된 점검:

- 원자료 주석/설명 행 제거
- 텍스트 키 정리와 숫자형 변환
- 기업유형, 초임급여 구간, 산업유형, 직무별 자격증 경쟁형 구성 합계 검산
- 통합 샘플 merge 후 pandas NULL 없음 확인
- 원천 코드형 `산업분류=결측값`, `직무분류=미상`을 pandas NULL과 분리

남겨야 할 제한:

- 산업분류 `결측값`은 삭제하지 않고 미상 비율로 주석 처리한다.
- 특히 교육 계열은 산업분류 미상 비율이 높아 산업 믹스 해석에 제한이 있다.
- 직무별 자격증은 계열/산업과 직접 조인하지 않는다.
- `직무분류 -> 산업분류` 매핑표는 후보표이며 수동 검토 전에는 분석 피처로 사용하지 않는다.

## 산출물

- 노트북: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0_계열별 기업, 산업, 자격증, 초임급여, 직무별.IPYNB`
- 직무-산업 후보 매핑표: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/tables/23_cert_job_to_industry_mapping_candidates.csv`
- 리포트: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0_major_cleaning_decision_report.md`
